# 🛠️ Environment Setup & Initialization
This cell initializes the environment, installs missing utilities, and imports all necessary dependencies. Centralizing imports here prevents duplicate import statements later and keeps the notebook professional and clean.

In [1]:
from shapely.geometry import Point, LineString
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu
%pip install python-dotenv -q

from datetime import timedelta
from dotenv import load_dotenv
from rasterio.windows import Window
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from shapely.geometry import box, mapping
from shapely.geometry import shape
from shapely.ops import unary_union
from skimage.feature import graycomatrix, graycoprops
import os
import glob
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
import datetime
import fiona
import hashlib
import pyproj
import rasterio
import re
import requests
import shutil
import time
import warnings

import warnings
warnings.filterwarnings('ignore')
print("Environment complete. All libraries loaded.")


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Environment complete. All libraries loaded.


# 📂 Layer & Table Extraction
This step extracts geographical layers and their attribute tables from the master GeoPackage (`all_layers.gpkg`). It automatically categorizes them into strictly tabular data (CSV) and spatial geometric layers (GPKG), setting the foundation for mapping and API analysis.

In [2]:
# This block extracts spatial layers and tables from the master GeoPackage.
# It categorizes them into spatial data (GeoDataFrames) and tabular data,
# and saves them separately into designated directories.

# Constants
SOURCE_GPKG = "qgis/all_layers.gpkg"
LAYERS_DIR = "extracted_layers"
TABLES_DIR = "extracted_tables"

# Ensure both target directories exist
os.makedirs(LAYERS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# Get all layer names inside the GeoPackage
layer_names = fiona.listlayers(SOURCE_GPKG)

print(f"Exporting {len(layer_names)} layers...")

for name in layer_names:
    # 1. Read the specific layer
    gdf = gpd.read_file(SOURCE_GPKG, layer=name)
    
    # 2. Check if the layer is spatial (GeoDataFrame)
    if isinstance(gdf, gpd.GeoDataFrame) and gdf.geometry is not None:
        # Save the spatial layer to 'extracted_layers'
        spatial_path = os.path.join(LAYERS_DIR, f"{name}.gpkg")
        gdf.to_file(spatial_path, driver="GPKG")
        
        # Save the attribute table to 'extracted_tables'
        table_path = os.path.join(TABLES_DIR, f"{name}.csv")
        gdf.drop(columns='geometry').to_csv(table_path, index=False)
        print(f" [SPATIAL] {name}")
    else:
        # 3. It's a non-spatial table (DataFrame)
        table_path = os.path.join(TABLES_DIR, f"{name}.csv")
        gdf.to_csv(table_path, index=False)
        print(f" [TABLE ONLY] {name}")

print("\nAll layers and tables have been successfully categorized and saved.")

Exporting 54 layers...
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Gadasj_haemek_2010
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Eden_d_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Eden_hei_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — EHI_tet_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Geva_13_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Kfar_horash_2011
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Mesilot_um_savisa_2011
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_hamra_vav
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_shokef_alef
 [SPATIAL] Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_

# 🧹 Data Cleaning & Standardization
Here, we standardize the spelling of the core target variable (`Aleket_cnt`) across all extracted tables. This handles multiple known variations and typos, casting the data into clean, numeric vectors formatted for analytical processing.

In [3]:
# This block standardizes the required 'Aleket_cnt' column across all extracted tables.
# It matches various known spelling variations to our target column name,
# ensures it is a numeric type, handles missing values, and saves the cleaned tables.

# Configuration
INPUT_DIR = "extracted_tables"
OUTPUT_DIR = "clean_tables"
TARGET_COL = "Aleket_cnt"

# Comprehensive list of variations (all lowercase for matching)
# Added 'aleket_cnt' to the list to ensure correct columns are captured
VARIATIONS = [
    "aleket_cnt", "aleket", "aleket_cou", "attrib_1", "aleket_fin", 
    "aleketlvl", "alaketlv", "aleketleve", "alaket", "aleket_count", 
    "numflowers", "count", "amount","_10_7_25","_10_8_25","_24_7_25","_3_8_25","_16_7_25"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting standardization...")

for filename in os.listdir(INPUT_DIR):
    if not filename.endswith(".csv"):
        continue
        
    path = os.path.join(INPUT_DIR, filename)
    try:
        df = pd.read_csv(path)
        
        # Mapping of lowercase names to original column names
        lower_to_orig = {col.lower(): col for col in df.columns}
        
        match_found = None
        
        # 1. Check if the target column (or its lowercase version) already exists
        if TARGET_COL in df.columns:
            match_found = TARGET_COL
        else:
            # 2. Search for variations
            for var in VARIATIONS:
                if var in lower_to_orig:
                    match_found = lower_to_orig[var]
                    # Rename it to the standard version
                    df = df.rename(columns={match_found: TARGET_COL})
                    break
        
        if match_found:
            # 3. Clean and convert to numeric (Crucial even for "correct" tables)
            df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
            
            # 4. Handle NaNs and cast to integer
            df[TARGET_COL] = df[TARGET_COL].fillna(0).astype(int)
            
            # 5. Specific normalization for GDD monitor files
            if filename in ["Monitor_GDD714_Broomrape_2025-06-05.csv", "Monitor_GDD562_Broomrape_2025-05-25.csv"]:
                conditions = [
                    df[TARGET_COL] >= 10,
                    (df[TARGET_COL] >= 2) & (df[TARGET_COL] <= 9),
                    df[TARGET_COL] == 1
                ]
                choices = [3, 2, 1]
                df[TARGET_COL] = np.select(conditions, choices, default=df[TARGET_COL])
            
            # Save the file to the output directory
            out_path = os.path.join(OUTPUT_DIR, filename)
            df.to_csv(out_path, index=False)
            print(f" [OK] {filename} (Column: {match_found})")
        else:
            print(f" [SKIP] {filename}: No valid column found.")
            
    except Exception as e:
        print(f"  [ERROR] Feature extraction failed for {chip_path}: {e}")
        print(f" [ERROR] {filename}: {e}")

print(f"\nStandardization complete. Files are in '{OUTPUT_DIR}'.")

Starting standardization...
 [OK] Boundary_Fields_Addition_2010-12 — Eden_d_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Eden_hei_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — EHI_tet_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Gadasj_haemek_2010.csv (Column: Attrib_1)
 [OK] Boundary_Fields_Addition_2010-12 — Geva_13_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Kfar_horash_2011.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011.csv (Column: Aleket_cnt)
 [OK] Boundary_Fields_Addition_2010-12 — Mesilot_um_

# 🗓️ Definition of Sample Dates
We define a manual lookup dictionary mapping each specific spatial field to its corresponding ground-truth sampling date. This date anchors our temporal analysis window for satellite data.

In [2]:
# Definition of sample dates for each field
sample_dates = {
    "Broomrape_Sampling_Gonen_2025-06-30": "30.06.25",
    "Boundary_Fields_Baseline_2011 — Yagur_buki": "21.07.11",
    "Boundary_Fields_Baseline_2011 — Yaen_kavnoa": "25.07.11",
    "Boundary_Fields_Baseline_2011 — Reem_damon_5": "14.07.11",
    "Boundary_Fields_Baseline_2011 — Reem_bialik_west": "14.07.11",
    "Boundary_Fields_Baseline_2011 — Hulata_krad_3_4": "02.08.11",
    "Boundary_Fields_Baseline_2011 — EHI_ablek_south": "30.06.11",
    "Boundary_Fields_Baseline_2011 — HavatGadash_30": "07.08.11",
    "Boundary_Fields_Baseline_2011 — Eden_gimel": "03.06.11",
    "Monitor_GDD562_Broomrape_2025-05-25": "25.05.25",
    "Boundary_Fields_Addition_2010-12 — Yaen_31_2012": "05.07.12",
    "Boundary_Fields_Addition_2010-12 — Sde_Nachum_2012": "21.06.12",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_tukan_d_2012": "06.06.12",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_Kemach_W_2012": "09.06.11",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_kemach_E": "09.06.11",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_hamra_hey_2012": "30.05.12",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_shokef_alef": "02.06.11",
    "Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_hamra_vav": "02.06.11",
    "Boundary_Fields_Addition_2010-12 — Mesilot_um_savisa_2011": "03.06.11",
    "Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011": "22.06.11",
    "Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011": "10.06.11",
    "Boundary_Fields_Addition_2010-12 — Geva_13_2012": "28.06.12",
    "Boundary_Fields_Addition_2010-12 — Kfar_horash_2011": "01.07.11",
    "Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012": "20.07.12",
    "Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011": "07.08.11",
    "Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012": "26.07.12",
    "Boundary_Fields_Addition_2010-12 — Eden_hei_2012": "15.06.12",
    "Boundary_Fields_Addition_2010-12 — EHI_tet_2012": "02.07.12",
    "Boundary_Fields_Addition_2010-12 — Eden_d_2012": "14.06.12",
    "Monitor_GDD714_Broomrape_2025-06-05": "05.06.25",
    "Boundary_Fields_Addition_2010-12 — Gadasj_haemek_2010": "05.07.10",
    "fields_2010 — Dan_BeitHilel_east": "10.08.10",
    "fields_2010 — Dan_Namme4east_1_points": "25.08.10",
    "fields_2010 — Dan_Namme4east_2_points": "25.08.10",
    "fields_2010 — EHI_emek_d_points": "01.06.10",
    "fields_2010 — EHI_hey_points": "01.06.10",
    "fields_2010 — Geva": "01.06.10",
    "fields_2010 — havateden_hey_points": "09.06.10",
    "fields_2010 — HavatGadash": "28.07.10",
    "fields_2010 — Hulata_west_points": "17.08.10",
    "fields_2010 — Yagur72": "12.07.10",
    "fields_2010 — Yagur90": "14.07.10",
    "fields_2010 — Yifaat_Kishon1_points": "21.07.10",
    "fields_2010 — Yifaat_vav": "18.07.10",
    "fields_2010 — kfarHahoresh_37": "18.07.10",
    "Havat_mataim_7.7.2025": "07.07.25",
    "manara_160725": "16.07.25",
    "rimonim_100725": "10.07.25",
    "Sasa_7_3.8.25": "03.08.25",
    "Kfar_sald_shaked_east_240725": "24.07.25",
    "Shamir_har_100725": "24.07.25",
    "hazorea_100825": "10.08.25",
    "gonen_maagar_220725": "22.07.25"
}


# 🔍 Sample Date Validation
This validation block runs a cross-check between all clean files in the directory and the explicitly declared `sample_dates`. This pinpoints any missing mappings and safely reports mismatched data points before fetching expensive API assets.

In [5]:
# Check that all files in 'clean_tables' directory have a corresponding sample date
missing_dates = []
matched_files = 0
total_files = 0

print("Checking files against sample dates...")
for filename in os.listdir(OUTPUT_DIR):
    if not filename.endswith(".csv"):
        continue

    total_files += 1
    # Strip '.csv' extension to match the field name keys
    field_name = filename[:-4]

    if field_name in sample_dates:
        matched_files += 1
    else:
        missing_dates.append(field_name)

print(f"\nSummary: {matched_files}/{total_files} files matched.")
if missing_dates:
    print("\nMissing dates for the following fields:")
    for pf in missing_dates:
        print(f" - {pf}")
else:
    print("\nAll fields have a matching sample date!")


Checking files against sample dates...

Summary: 53/53 files matched.

All fields have a matching sample date!


# 📡 Boolean Temporal Coverage (Planet Quick-Search)
We query the Planet Labs Data API using `quick-search`. This block generates a pure Boolean matrix indicating whether usable satellite imagery exists over specific geographic bounding boxes leading up to the sample dates. It maps the presence of data across 8 temporal bins.

In [6]:
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

def generate_boolean_coverage():
    """
    Generates a Boolean temporal coverage matrix for spatial datasets 
    using the Planet Data API. Checks for intersections with quick-search metadata.
    """
    CLEAN_TABLES_DIR = "clean_tables"
    EXTRACTED_LAYERS_DIR = "extracted_layers"
    OUTPUT_DIR = "coverage_matrices"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
    
    if not PLANET_API_KEY:
        print("[ERROR] PLANET_API_KEY not found in environment.")
        return

    headers = {"Content-Type": "application/json"}
    auth = (PLANET_API_KEY, "")

    try:
        sample_dates
    except NameError:
        print("[ERROR] 'sample_dates' dictionary is not defined in the current scope.")
        return

    for field_name, s_date_str in sample_dates.items():
        print(f"\nProcessing: {field_name}")
        
        # 1. Parse sample date and calculate 56-day window
        if "." in s_date_str:
            end_date = pd.to_datetime(s_date_str, format="%d.%m.%y")
        else:
            end_date = pd.to_datetime(s_date_str, format="%Y-%m-%d")
            
        start_date = end_date - timedelta(days=56)
        
        # 2. Load geometry table and clean table
        csv_path = os.path.join(CLEAN_TABLES_DIR, f"{field_name}.csv")
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")
        
        if not os.path.exists(csv_path):
            print(f"  [SKIP] CSV not found: {csv_path}")
            continue
            
        df_table = pd.read_csv(csv_path)
        
        if not os.path.exists(gpkg_path):
            print(f"  [WARNING] Spatial GPKG not found: {gpkg_path}.")
            continue
            
        gdf_spatial = gpd.read_file(gpkg_path)
        
        # Determine ID column
        id_col = 'id' if 'id' in df_table.columns else df_table.columns[0]
        if id_col not in gdf_spatial.columns:
            if len(gdf_spatial) == len(df_table):
                gdf_spatial[id_col] = df_table[id_col].values
            else:
                print(f"  [SKIP] ID column mismatch and different lengths for {field_name}.")
                continue
                
        if gdf_spatial.crs is None or gdf_spatial.crs.to_epsg() != 4326:
            gdf_spatial = gdf_spatial.to_crs(epsg=4326)
            
        # 3. Calculate bounding box block
        bbox = gdf_spatial.total_bounds
        bbox_geom = {
            "type": "Polygon",
            "coordinates": [[[bbox[0], bbox[1]], [bbox[2], bbox[1]], 
                             [bbox[2], bbox[3]], [bbox[0], bbox[3]], 
                             [bbox[0], bbox[1]]]]
        }
        
        # 4. Construct Planet API metadata request
        geom_filter = {"type": "GeometryFilter", "field_name": "geometry", "config": bbox_geom}
        date_filter = {
            "type": "DateRangeFilter", 
            "field_name": "acquired", 
            "config": {
                "gte": start_date.strftime("%Y-%m-%dT00:00:00.000Z"),
                "lte": end_date.strftime("%Y-%m-%dT23:59:59.999Z")
            }
        }
        cloud_filter = {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lt": 0.20}}
        combined_filter = {"type": "AndFilter", "config": [geom_filter, date_filter, cloud_filter]}
        
        search_request = {
            "item_types": ["PSScene", "REOrthoTile", "REScene"],
            "filter": combined_filter
        }
        
        # 5. Execute API request with Pagination
        features = []
        res = requests.post(SEARCH_URL, json=search_request, headers=headers, auth=auth)
        
        if res.status_code != 200:
            print(f"  [ERROR] API request failed: {res.status_code} - {res.text}")
            continue
            
        data = res.json()
        features.extend(data.get('features', []))
        
                # Changed condition to ensure _next is not None
        while '_links' in data and data['_links'].get('_next'):
            res = requests.get(data['_links']['_next'], headers=headers, auth=auth)

            if res.status_code == 200:
                data = res.json()
                features.extend(data.get('features', []))
            else:
                break
                
        print(f"  Retrieved {len(features)} valid scenes metadata.")
        
        # 6. Local Processing & Binning
        if not features:
            out_df = pd.DataFrame({id_col: gdf_spatial[id_col]})
            for i in range(1, 9): out_df[f'image{i}'] = 0
        else:
            scenes_data = []
            for feat in features:
                acq_dt = pd.to_datetime(feat['properties']['acquired']).tz_localize(None)
                days_diff = (end_date - acq_dt).days
                
                if days_diff < 0 or days_diff > 56: continue
                
                bin_idx = 8 - (days_diff // 7)
                if bin_idx < 1: bin_idx = 1
                
                scenes_data.append({
                    'scene_id': feat['id'],
                    'bin': f'image{bin_idx}',
                    'geometry': shape(feat['geometry'])
                })
                
            if not scenes_data:
                out_df = pd.DataFrame({id_col: gdf_spatial[id_col]})
                for i in range(1, 9): out_df[f'image{i}'] = 0
            else:
                gdf_scenes = gpd.GeoDataFrame(scenes_data, geometry='geometry', crs="EPSG:4326")
                joined = gpd.sjoin(gdf_spatial, gdf_scenes, how="left", predicate="intersects")
                
                out_matrix = []
                for pt_id, group in joined.groupby(id_col):
                    row = {id_col: pt_id}
                    for i in range(1, 9): row[f'image{i}'] = 0
                    
                    bins_found = group['bin'].dropna().unique()
                    for b in bins_found: row[b] = 1
                    out_matrix.append(row)
                    
                out_df = pd.DataFrame(out_matrix)
                
                for i in range(1, 9):
                    if f'image{i}' not in out_df.columns: out_df[f'image{i}'] = 0
                    
        # 7. Format final CSV
        cols_order = [id_col] + [f'image{i}' for i in range(1, 9)]
        out_df = out_df[cols_order]
        
        out_path = os.path.join(OUTPUT_DIR, f"{field_name}_coverage.csv")
        out_df.to_csv(out_path, index=False)
        print(f"  [SAVED] {out_path}")

print("\nInitialization Complete. Executing script...")
if __name__ == "__main__":
    generate_boolean_coverage()


Initialization Complete. Executing script...

Processing: Broomrape_Sampling_Gonen_2025-06-30
  Retrieved 62 valid scenes metadata.
  [SAVED] coverage_matrices\Broomrape_Sampling_Gonen_2025-06-30_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Yagur_buki
  Retrieved 21 valid scenes metadata.
  [SAVED] coverage_matrices\Boundary_Fields_Baseline_2011 — Yagur_buki_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Yaen_kavnoa
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices\Boundary_Fields_Baseline_2011 — Yaen_kavnoa_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Reem_damon_5
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices\Boundary_Fields_Baseline_2011 — Reem_damon_5_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Reem_bialik_west
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices\Boundary_Fields_Baseline_2011 — Reem_bialik_west_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Hulata_krad_3_4

# 📅 High-Resolution Temporal Coverage Generation
Similar to the Boolean check, this phase queries the Planet API but explicitly documents the **exact acquisition dates** of available high-resolution scenes. This ensures temporally precise mapping between satellite observables and ground-truth phenomena.

In [33]:
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

def generate_temporal_coverage():
    """
    Generates a temporal coverage matrix for spatial datasets 
    using the Planet Data API. Outputs exact dates of available images per point.
    """
    CLEAN_TABLES_DIR = "clean_tables"
    EXTRACTED_LAYERS_DIR = "extracted_layers"
    OUTPUT_DIR = "coverage_matrices_dates"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
    
    if not PLANET_API_KEY:
        print("[ERROR] PLANET_API_KEY not found in environment.")
        return

    headers = {"Content-Type": "application/json"}
    auth = (PLANET_API_KEY, "")

    try:
        sample_dates
    except NameError:
        print("[ERROR] 'sample_dates' dictionary is not defined in the current scope.")
        return

    for field_name, s_date_str in sample_dates.items():
        print(f"\nProcessing: {field_name}")
        
        # 1. Parse sample date and calculate 56-day window
        if "." in s_date_str:
            end_date = pd.to_datetime(s_date_str, format="%d.%m.%y")
        else:
            end_date = pd.to_datetime(s_date_str, format="%Y-%m-%d")
            
        start_date = end_date - timedelta(days=56)
        
        # 2. Load geometry table and clean table
        csv_path = os.path.join(CLEAN_TABLES_DIR, f"{field_name}.csv")
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")
        
        if not os.path.exists(csv_path):
            print(f"  [SKIP] CSV not found: {csv_path}")
            continue
            
        df_table = pd.read_csv(csv_path)
        
        if not os.path.exists(gpkg_path):
            print(f"  [WARNING] Spatial GPKG not found: {gpkg_path}.")
            continue
            
        gdf_spatial = gpd.read_file(gpkg_path)
        
        # Determine ID column
        id_col = 'id' if 'id' in df_table.columns else df_table.columns[0]
        if id_col not in gdf_spatial.columns:
            if len(gdf_spatial) == len(df_table):
                gdf_spatial[id_col] = df_table[id_col].values
            else:
                print(f"  [SKIP] ID column mismatch and different lengths for {field_name}.")
                continue
                
        if gdf_spatial.crs is None or gdf_spatial.crs.to_epsg() != 4326:
            gdf_spatial = gdf_spatial.to_crs(epsg=4326)
            
        # 3. Calculate bounding box block
        bbox = gdf_spatial.total_bounds
        bbox_geom = {
            "type": "Polygon",
            "coordinates": [[[bbox[0], bbox[1]], [bbox[2], bbox[1]], 
                             [bbox[2], bbox[3]], [bbox[0], bbox[3]], 
                             [bbox[0], bbox[1]]]]
        }
        
        # 4. Construct Planet API metadata request
        geom_filter = {"type": "GeometryFilter", "field_name": "geometry", "config": bbox_geom}
        date_filter = {
            "type": "DateRangeFilter", 
            "field_name": "acquired", 
            "config": {
                "gte": start_date.strftime("%Y-%m-%dT00:00:00.000Z"),
                "lte": end_date.strftime("%Y-%m-%dT23:59:59.999Z")
            }
        }
        cloud_filter = {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lt": 0.20}}
        combined_filter = {"type": "AndFilter", "config": [geom_filter, date_filter, cloud_filter]}
        
        search_request = {
            "item_types": ["PSScene", "REOrthoTile", "REScene"],
            "filter": combined_filter
        }
        
        # 5. Execute API request with Pagination
        features = []
        res = requests.post(SEARCH_URL, json=search_request, headers=headers, auth=auth)
        
        if res.status_code != 200:
            print(f"  [ERROR] API request failed: {res.status_code} - {res.text}")
            continue
            
        data = res.json()
        features.extend(data.get('features', []))
        
        while '_links' in data and data['_links'].get('_next'):
            res = requests.get(data['_links']['_next'], headers=headers, auth=auth)
            if res.status_code == 200:
                data = res.json()
                features.extend(data.get('features', []))
            else:
                break
                
        print(f"  Retrieved {len(features)} valid scenes metadata.")
        
        # 6. Local Processing & Date Extraction
        if not features:
            out_df = pd.DataFrame({id_col: gdf_spatial[id_col]})
        else:
            scenes_data = []
            for feat in features:
                acq_dt = pd.to_datetime(feat['properties']['acquired']).tz_localize(None)
                days_diff = (end_date - acq_dt).days
                
                if days_diff < 0 or days_diff > 56: continue
                
                # Format the date as DD/MM/YYYY
                date_str = acq_dt.strftime("%d/%m/%Y")
                
                scenes_data.append({
                    'scene_id': feat['id'],
                    'acq_date': date_str,
                    'geometry': shape(feat['geometry'])
                })
                
            if not scenes_data:
                out_df = pd.DataFrame({id_col: gdf_spatial[id_col]})
            else:
                gdf_scenes = gpd.GeoDataFrame(scenes_data, geometry='geometry', crs="EPSG:4326")
                joined = gpd.sjoin(gdf_spatial, gdf_scenes, how="left", predicate="intersects")
                
                out_matrix = []
                max_images = 0
                
                for pt_id, group in joined.groupby(id_col):
                    row = {id_col: pt_id}
                    
                    # Drop NaN values, isolate unique dates, and sort chronologically
                    unique_dates = group['acq_date'].dropna().unique()
                    sorted_dates = sorted(unique_dates, key=lambda d: pd.to_datetime(d, format="%d/%m/%Y"))
                    
                    # Populate the row with image1, image2, etc.
                    for i, date_val in enumerate(sorted_dates, start=1):
                        row[f'image{i}'] = date_val
                        
                    max_images = max(max_images, len(sorted_dates))
                    out_matrix.append(row)
                    
                out_df = pd.DataFrame(out_matrix)
                
                # Standardize column structure up to the maximum number of images found
                if max_images > 0:
                    cols_order = [id_col] + [f'image{i}' for i in range(1, max_images + 1)]
                    out_df = out_df.reindex(columns=cols_order)
                    
        # 7. Format final CSV
        out_path = os.path.join(OUTPUT_DIR, f"{field_name}_coverage.csv")
        out_df.to_csv(out_path, index=False)
        print(f"  [SAVED] {out_path}")

print("\nInitialization Complete. Executing script...")
if __name__ == "__main__":
    generate_temporal_coverage()


Initialization Complete. Executing script...

Processing: Broomrape_Sampling_Gonen_2025-06-30
  Retrieved 62 valid scenes metadata.
  [SAVED] coverage_matrices_dates\Broomrape_Sampling_Gonen_2025-06-30_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Yagur_buki
  Retrieved 21 valid scenes metadata.
  [SAVED] coverage_matrices_dates\Boundary_Fields_Baseline_2011 — Yagur_buki_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Yaen_kavnoa
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices_dates\Boundary_Fields_Baseline_2011 — Yaen_kavnoa_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Reem_damon_5
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices_dates\Boundary_Fields_Baseline_2011 — Reem_damon_5_coverage.csv

Processing: Boundary_Fields_Baseline_2011 — Reem_bialik_west
  Retrieved 12 valid scenes metadata.
  [SAVED] coverage_matrices_dates\Boundary_Fields_Baseline_2011 — Reem_bialik_west_coverage.csv

Processing: Boundary_Fields_B

# 🛒 Planet API Ordering Pipeline
This section bundles geographically validated assets into a synchronized historical order payload. By mathematically validating non-intersecting geometries and confirming temporal coverage, it optimizes the global order placement directly via Planet's Orders API v2.

In [17]:
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"

HEADERS = {"Content-Type": "application/json"}
AUTH = (PLANET_API_KEY, "")

EXTRACTED_LAYERS_DIR = "extracted_layers"

def place_global_historical_order():
    historical_fields = {k: v for k, v in sample_dates.items() if pd.to_datetime(v, dayfirst=True).year < 2013}
    
    print(f"Scanning {len(historical_fields)} fields to build 1 global order...\n")

    global_item_ids = set()
    global_geometries = [] 
    processed_fields = 0

    for field_name, s_date_str in historical_fields.items():
        sample_date = pd.to_datetime(s_date_str, dayfirst=True)
        start_date = sample_date - timedelta(days=56)
        
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")
        if not os.path.exists(gpkg_path):
            continue

        gdf_layer = gpd.read_file(gpkg_path)
        if gdf_layer.crs is None or gdf_layer.crs.to_epsg() != 4326:
            gdf_layer = gdf_layer.to_crs(epsg=4326)
            
        bbox = gdf_layer.total_bounds 
        field_box = box(*bbox)
        bbox_geom = mapping(field_box)

        # 1. Metadata Search
        search_request = {
            # ---> FIX: Strictly limited to REOrthoTile to ensure Surface Reflectance support <---
            "item_types": ["REOrthoTile"], 
            "filter": {
                "type": "AndFilter",
                "config": [
                    {"type": "GeometryFilter", "field_name": "geometry", "config": bbox_geom},
                    {"type": "DateRangeFilter", "field_name": "acquired", "config": {
                        "gte": start_date.strftime("%Y-%m-%dT00:00:00.000Z"),
                        "lte": sample_date.strftime("%Y-%m-%dT23:59:59.999Z")
                    }},
                    {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lt": 0.50}} 
                ]
            }
        }

        res = requests.post(SEARCH_URL, json=search_request, headers=HEADERS, auth=AUTH)
        if res.status_code != 200:
            continue
            
        features = res.json().get('features', [])
        features.sort(key=lambda x: pd.to_datetime(x['properties']['acquired']))
        
        # 2. Temporal Pruning
        selected_item_ids = []
        selected_dates = []
        
        for feat in features:
            acq_dt = pd.to_datetime(feat['properties']['acquired']).tz_localize(None)
            
            if not selected_dates:
                selected_dates.append(acq_dt)
                selected_item_ids.append(feat['id'])
            else:
                if (acq_dt - selected_dates[-1]).days >= 4:
                    selected_dates.append(acq_dt)
                    selected_item_ids.append(feat['id'])

        if len(selected_item_ids) >= 2:
            global_item_ids.update(selected_item_ids)
            global_geometries.append(field_box)
            processed_fields += 1
            print(f"  [ADDED] {field_name}: {len(selected_item_ids)} mathematically valid scenes.")

    # 3. Submit the 1 Global Order
    if not global_item_ids:
        print("\nNo fields met the strict REOrthoTile criteria. No order placed.")
        return

    print("\nValidating and dissolving geometries to prevent self-intersections...")
    merged_geometry = unary_union(global_geometries)
    valid_aoi_geojson = mapping(merged_geometry)

    print(f"\nConsolidated Order Summary:")
    print(f" - Fields included: {processed_fields}")
    print(f" - Total unique scenes: {len(global_item_ids)}")
    
    order_payload = {
        "name": "Global_Historical_Thesis_Order",
        "products": [
            {
                "item_ids": list(global_item_ids),
                "item_type": "REOrthoTile",
                "product_bundle": "analytic_sr"
            }
        ],
        "tools": [
            {
                "clip": {
                    "aoi": valid_aoi_geojson 
                }
            }
        ]
    }

    print("\nSubmitting 1 global order to Planet API...")
    order_res = requests.post(ORDERS_URL, json=order_payload, headers=HEADERS, auth=AUTH)
    
    if order_res.status_code == 202:
        order_id = order_res.json().get('id')
        print(f"[SUCCESS] Order Placed! Order ID: {order_id}")
    else:
        print(f"[ORDER ERROR] Failed to place order: {order_res.status_code} - {order_res.text}")
        
if __name__ == "__main__":
    place_global_historical_order()

Scanning 42 fields to build 1 global order...

  [ADDED] Boundary_Fields_Baseline_2011 — Yagur_buki: 5 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — Yaen_kavnoa: 4 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — Reem_damon_5: 5 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — Reem_bialik_west: 5 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — Hulata_krad_3_4: 4 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — EHI_ablek_south: 5 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — HavatGadash_30: 4 mathematically valid scenes.
  [ADDED] Boundary_Fields_Baseline_2011 — Eden_gimel: 4 mathematically valid scenes.
  [ADDED] Boundary_Fields_Addition_2010-12 — Yaen_31_2012: 2 mathematically valid scenes.
  [ADDED] Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_Kemach_W_2012: 5 mathematically valid scenes.
  [ADDED] Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_

# ⬇️ Satellite Imagery Download
After a successful API order placement, this script intelligently polls the Planet API until the status is `SUCCESS`. Once ready, it securely and sequentially streams the target imagery (`.tif` files) to the local disk, bypassing structural limitations of large monolithic downloads.

In [20]:
# --- Configuration ---
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

# Paste your specific Order ID here
ORDER_ID = "c6b43940-9938-4d97-b9c6-481602703429" 
DOWNLOAD_DIR = "data/raw_chips"

# Ensure the target directory exists
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

ORDERS_URL = f"https://api.planet.com/compute/ops/orders/v2/{ORDER_ID}"
AUTH = (PLANET_API_KEY, "")

def download_planet_order():
    print(f"Checking status for Order: {ORDER_ID}")
    
    # 1. Poll the API until the order is ready
    while True:
        res = requests.get(ORDERS_URL, auth=AUTH)
        if res.status_code != 200:
            print(f"[ERROR] Could not fetch order status: {res.status_code} - {res.text}")
            return
            
        order_data = res.json()
        state = order_data.get('state')
        
        print(f"Current Status: {state.upper()}")
        
        if state == 'success':
            break
        elif state in ['failed', 'partial']:
            print("[WARNING] Order did not complete fully. Check Planet UI for details.")
            if state == 'failed':
                return
            break # If partial, we can still download what succeeded
        else:
            print("  -> Order is still processing. Waiting 30 seconds...")
            time.sleep(30)

    # 2. Extract download links
    results = order_data.get('_links', {}).get('results', [])
    if not results:
        print("No download links found in the order response.")
        return

    # Filter only for .tif files to save bandwidth and disk space (ignore .xml and manifests)
    tif_files = [r for r in results if r['name'].endswith('.tif')]
    
    print(f"\nFound {len(tif_files)} .tif files ready for download.")
    
    # 3. Download files sequentially
    downloaded_count = 0
    for file_info in tif_files:
        download_url = file_info['location']
        
        # Planet's file names are long and include folders. We just want the base filename.
        base_filename = os.path.basename(file_info['name'])
        save_path = os.path.join(DOWNLOAD_DIR, base_filename)
        
        # Skip if we already downloaded it
        if os.path.exists(save_path):
            print(f"  [SKIP] Already exists: {base_filename}")
            downloaded_count += 1
            continue
            
        print(f"  Downloading: {base_filename} ...", end="", flush=True)
        
        try:
            # Stream the download to handle larger files without crashing RAM
            dl_res = requests.get(download_url, stream=True)
            dl_res.raise_for_status()
            
            with open(save_path, 'wb') as f:
                for chunk in dl_res.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(" Done.")
            downloaded_count += 1
            
        except Exception as e:
            print(f" FAILED. Error: {e}")

    print(f"\n[COMPLETE] Successfully secured {downloaded_count}/{len(tif_files)} files in {DOWNLOAD_DIR}/")

if __name__ == "__main__":
    download_planet_order()

Checking status for Order: c6b43940-9938-4d97-b9c6-481602703429
Current Status: SUCCESS

Found 72 .tif files ready for download.
  Downloading: 3654124_2010-04-16_RE2_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654124_2010-04-16_RE2_3A_udm_clip.tif ... Done.
  Downloading: 3654124_2011-05-16_RE1_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654124_2011-05-16_RE1_3A_udm_clip.tif ... Done.
  Downloading: 3654024_2011-05-19_RE4_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654024_2011-05-19_RE4_3A_udm_clip.tif ... Done.
  Downloading: 3654024_2011-06-06_RE3_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654024_2011-06-06_RE3_3A_udm_clip.tif ... Done.
  Downloading: 3654123_2011-05-16_RE1_3A_udm_clip.tif ... Done.
  Downloading: 3654123_2011-05-16_RE1_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654124_2012-06-16_RE3_3A_udm_clip.tif ... Done.
  Downloading: 3654124_2012-06-16_RE3_3A_Analytic_SR_clip.tif ... Done.
  Downloading: 3654124_2010-04-12_RE3_3A_Analytic_SR_cl

# 🧬 Pixel-Level Feature Extraction & Long-Format Construction
This is the core numerical processing engine. We dynamically parse downloaded multi-spectral imagery. For every sampled geographic point, we calculate:
* **Vegetation Indices:** `NDVI`, `GNDVI`, `NDRE`, `Green/Red Ratio`
* **GLCM Textural Features:** `Entropy`, `Homogeneity`, `ASM`
The final dataset maps physiological and structural dynamics across backward-looking temporal steps into a Long format design perfectly suited for statistical modeling.

In [7]:
# --- Constants & Paths ---
RAW_CHIPS_DIR = "data/raw_chips" 
CLEAN_TABLES_DIR = "clean_tables"
EXTRACTED_LAYERS_DIR = "extracted_layers"
OUTPUT_DIR = "field_temporal_datasets_long" 

os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_valid_chips_for_field(field_gdf, chips_dir, sample_date):
    """
    Scans chips to guarantee they intersect the field spatially AND 
    fall strictly within the 60 days leading up to the sample date.
    """
    valid_chips = []
    minx, miny, maxx, maxy = field_gdf.total_bounds
    
    centroid = field_gdf.geometry.unary_union.centroid
    cx, cy = centroid.x, centroid.y
    
    # Define the strict 60-day temporal window
    start_date = sample_date - timedelta(days=60)
    
    print(f"\n  [DIAGNOSTIC] Testing files for centroid (Lon: {cx:.4f}, Lat: {cy:.4f}).")
    print(f"  [DIAGNOSTIC] Valid temporal window: {start_date.strftime('%Y-%m-%d')} to {sample_date.strftime('%Y-%m-%d')}")
    files_checked = 0
    
    if not os.path.exists(chips_dir):
        print(f"  [ERROR] Directory does not exist: {chips_dir}")
        return valid_chips

    for f in os.listdir(chips_dir):
        if not f.endswith('.tif') or 'udm' in f.lower() or 'sr' not in f.lower():
            continue

        files_checked += 1
        chip_path = os.path.join(chips_dir, f)
        
        try:
            with rasterio.open(chip_path) as src:
                transformer = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                t_minx, t_miny = transformer.transform(minx, miny)
                t_maxx, t_maxy = transformer.transform(maxx, maxy)
                
                intersects = not (t_minx > src.bounds.right or t_maxx < src.bounds.left or
                                  t_miny > src.bounds.top or t_maxy < src.bounds.bottom)
                
                if intersects:
                    cx_proj, cy_proj = transformer.transform(cx, cy)
                    try:
                        py, px = src.index(cx_proj, cy_proj)
                        
                        val = src.read(1, window=Window(px, py, 1, 1)) 
                        
                        if val.size > 0 and val[0, 0] > 0:
                            # Your upgraded flexible date regex
                            match = re.search(r'(20\d{2}-?\d{2}-?\d{2})', f)
                            if match:
                                date_str = match.group(1)
                                acq_date = pd.to_datetime(date_str)
                                
                                # --- THE NEW TEMPORAL FILTER ---
                                if start_date <= acq_date <= sample_date:
                                    valid_chips.append({'path': chip_path, 'date': acq_date})
                                else:
                                    pass # Silently drop images outside the 60-day window to avoid spam
                            else:
                                print(f"    -> [FAIL Regex] Intersects, but can't find a valid date in filename: {f}")
                        else:
                            print(f"    -> [FAIL Pixel] Intersects, but pixel is 0 (Planet Padding): {f}")
                    except Exception as inner_e:
                        print(f"    -> [FAIL Index] Intersects, but centroid error on {f}: {inner_e}")
        except Exception as outer_e:
            print(f"    -> [FAIL Read] Could not open {f}: {outer_e}")
            
    print(f"  [DIAGNOSTIC] Checked {files_checked} files. Found {len(valid_chips)} valid chips within the 60-day window.")
    return valid_chips
    
def extract_features_at_point(chip_path: str, lon: float, lat: float) -> tuple[dict[str, float] | None, str]:
    """
    Extracts high-resolution satellite features at a specific geographic point.
    Spectral bands and indices are extracted from the EXACT 1x1 pixel.
    GLCM texture features are computed over a tight 3x3 surrounding neighborhood.
    """
    try:
        with rasterio.open(chip_path) as src:
            transformer = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            x_proj, y_proj = transformer.transform(lon, lat)
            
            # Spatial boundary check
            if not ((src.bounds.left - 100) <= x_proj <= (src.bounds.right + 100) and 
                    (src.bounds.bottom - 100) <= y_proj <= (src.bounds.top + 100)):
                return None, "WRONG_FIELD"
                
            py, px = src.index(x_proj, y_proj)
            
            # Ensure safe margins for the 3x3 texture window
            if py < 1 or px < 1 or py >= src.height - 1 or px >= src.width - 1:
                return None, "EDGE_PROXIMITY"
            
            # 1. SPECTRAL FEATURES: Extract from the exact 1x1 single pixel containing the point
            spectral_window = Window(px, py, 1, 1)
            spectral_data = src.read(window=spectral_window).astype(float)
            
            if spectral_data.shape[0] < 8:
                return None, "SENSOR_MISMATCH"
                
            # Extract scalar reflectance values from the single pixel
            green = spectral_data[3][0, 0]
            red = spectral_data[5][0, 0]
            red_edge = spectral_data[6][0, 0]
            nir = spectral_data[7][0, 0]
            
            if nir == 0:
                return None, "NODATA_COLLAR"
                
            # Perform direct scalar spectral index calculations
            mean_green = green
            
            denom_ndvi = nir + red
            mean_ndvi = (nir - red) / denom_ndvi if denom_ndvi != 0 else np.nan

            denom_gndvi = nir + green
            mean_gndvi = (nir - green) / denom_gndvi if denom_gndvi != 0 else np.nan

            mean_gr_ratio = green / red if red != 0 else np.nan

            denom_ndre = nir + red_edge
            mean_ndre = (nir - red_edge) / denom_ndre if denom_ndre != 0 else np.nan

            # 2. TEXTURE FEATURES: Read a tight 3x3 window strictly to enable GLCM math
            texture_window = Window(px - 1, py - 1, 3, 3)
            texture_data = src.read(window=texture_window).astype(float)
            nir_texture = texture_data[7]
            
            # Gray-level co-occurrence matrix math
            nir_8bit = np.clip(nir_texture / 10000.0 * 255, 0, 255).astype(np.uint8)
            angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
            glcm = graycomatrix(nir_8bit, distances=[1], angles=angles, symmetric=True, normed=True)
            
            homo = np.mean(graycoprops(glcm, 'homogeneity'))
            asm = np.mean(graycoprops(glcm, 'ASM'))
            
            glcm_avg = np.mean(glcm, axis=3)[:, :, 0]
            entropy = -np.sum(glcm_avg[glcm_avg > 0] * np.log2(glcm_avg[glcm_avg > 0]))

            features = {
                'green': mean_green, 'ndvi': mean_ndvi, 'gndvi': mean_gndvi,
                'green_red_ratio': mean_gr_ratio, 'ndre': mean_ndre,
                'homogeneity': homo, 'entropy': entropy, 'asm': asm
            }
            return features, "SUCCESS"
            
    except Exception as e:
        return None, f"EXCEPTION: {str(e)}"

        
def build_long_format_datasets():
    historical_fields = {k: v for k, v in sample_dates.items() if pd.to_datetime(v, dayfirst=True).year < 2013}
    print(f"Initiating Long Format extraction for {len(historical_fields)} fields...\n")

    for field_name, s_date_str in historical_fields.items():
        field_records = [] 
        sample_date = pd.to_datetime(s_date_str, dayfirst=True)
        
        csv_path = os.path.join(CLEAN_TABLES_DIR, f"{field_name}.csv")
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")

        if not all(os.path.exists(p) for p in [csv_path, gpkg_path]):
            print(f"[SKIP] Missing core files for {field_name}")
            continue

        df_table = pd.read_csv(csv_path)
        gdf_layer = gpd.read_file(gpkg_path).to_crs(epsg=4326)
        id_col = 'id' if 'id' in df_table.columns else df_table.columns[0]
        
        if id_col not in gdf_layer.columns:
            if len(gdf_layer) == len(df_table):
                gdf_layer[id_col] = df_table[id_col].values
            else:
                continue
                
        print(f"--- Processing Field: {field_name} ---")
        
        # Identify Valid Chips for the RapidEye Field
        valid_chips = get_valid_chips_for_field(gdf_layer, RAW_CHIPS_DIR, sample_date)
        
        if not valid_chips:
            print(f"  -> [WARNING] No valid overlapping images found in directory. Skipping field.")
            continue
            
        unique_dates = sorted(list(set([c['date'] for c in valid_chips])))
        
        # Temporal Pruning 
        selected_dates = []
        if unique_dates:
            selected_dates.append(unique_dates[0])
            for d in unique_dates[1:]:
                if (d - selected_dates[-1]).days >= 4:
                    selected_dates.append(d)
        
        if len(selected_dates) < 2:
            print(f"  -> [WARNING] Not enough temporally spaced images (< 2). Skipping field.")
            continue
            
        date_to_chips = {d: [] for d in selected_dates}
        for c in valid_chips:
            if c['date'] in selected_dates:
                date_to_chips[c['date']].append(c['path'])
        
        # Feature Extraction
        for idx, row in df_table.iterrows():
            pt_id = row[id_col]
            aleket = row['Aleket_cnt']
            
            pt_geom_series = gdf_layer[gdf_layer[id_col] == pt_id].geometry
            if pt_geom_series.empty:
                continue
            lat, lon = pt_geom_series.iloc[0].y, pt_geom_series.iloc[0].x
            
            for i, d in enumerate(selected_dates, start=1):
                date_str = d.strftime('%d/%m/%Y')
                record = {
                    'image': i, 'Point_ID': pt_id, 'Aleket_cnt': aleket,
                    'lat': lat, 'long': lon, 'date': date_str,
                    'Days_Before_Sample': (sample_date - d).days,
                    'ndvi': np.nan, 'gndvi': np.nan, 'green_red_ratio': np.nan,
                    'ndre': np.nan, 'entropy': np.nan, 'green': np.nan,
                    'homogeneity': np.nan, 'asm': np.nan
                }
                
                chip_paths = date_to_chips[d]
                success = False
                edge_proximity = False
                nodata_count = 0
                
                for chip_path in chip_paths:
                    features, error_msg = extract_features_at_point(chip_path, lon, lat)
                    
                    if features:
                        record.update(features)
                        success = True
                        break 
                    else:
                        if error_msg == "EDGE_PROXIMITY": edge_proximity = True
                        elif error_msg == "NODATA_COLLAR": nodata_count += 1
                            
                if not success:
                    if edge_proximity:
                        print(f"[FAIL] Pt {pt_id} | {date_str}: Edge Proximity. Cannot extract 3x3 window.")
                    elif nodata_count > 0:
                        print(f"[FAIL] Pt {pt_id} | {date_str}: NoData Zone. Point landed on black margin.")
            
                field_records.append(record)

        if field_records:
            final_df = pd.DataFrame(field_records)
            final_df = final_df.sort_values(by=['image', 'Point_ID'])
            out_path = os.path.join(OUTPUT_DIR, f"{field_name}_long_features.csv")
            final_df.to_csv(out_path, index=False)
            print(f"  -> Saved {len(final_df)} observation rows.")
        else:
            print(f"  -> No data generated.")

if __name__ == "__main__":
    build_long_format_datasets()

Initiating Long Format extraction for 42 fields...

--- Processing Field: Boundary_Fields_Baseline_2011 — Yagur_buki ---

  [DIAGNOSTIC] Testing files for centroid (Lon: 35.0846, Lat: 32.7404).
  [DIAGNOSTIC] Valid temporal window: 2011-05-22 to 2011-07-21
    -> [FAIL Pixel] Intersects, but pixel is 0 (Planet Padding): 3654123_2011-05-29_RE5_3A_Analytic_SR_clip.tif
  [DIAGNOSTIC] Checked 36 files. Found 7 valid chips within the 60-day window.
[FAIL] Pt 330 | 27/05/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 330 | 31/05/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 330 | 06/06/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 330 | 24/06/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 330 | 15/07/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 516 | 06/06/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 516 | 24/06/2011: Edge Proximity. Cannot extract 3x3 window.
[FAIL] Pt 517 | 06/06/2011: Edge Proximity. Cannot extr

# 🗺️ Spatial Visualization: Ground Truth Infestation
Generates single point-map visualizations over the geographic space, coloring data points categorically according to their ground-truth Aleket severity values. This confirms logical clustering and initial spatial patterns.

In [8]:
def generate_single_infestation_map(csv_path, output_dir="visualizations_infestation"):
    """
    Generates a single spatial point map of the field colored by Aleket_cnt.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    
    # Isolate unique spatial points since Aleket_cnt does not change over time
    unique_points = df.drop_duplicates(subset=['Point_ID']).dropna(subset=['lat', 'long', 'Aleket_cnt'])
    
    if unique_points.empty:
        print(f"[SKIP] No valid spatial data for {field_name}")
        return

    fig, ax = plt.subplots(figsize=(10, 10))    
    color_map = {
        0: ('#2CA02C', 'No Infestation (0)'),      # Green
        1: ('#FFD700', 'Low Infestation (1)'),      # Gold
        2: ('#FF7F0E', 'Medium Infestation (2)'),   # Orange
        3: ('#D62728', 'High Infestation (3)')      # Red
    }

    for score, (color, label) in color_map.items():
        points = unique_points[unique_points['Aleket_cnt'] == score]
        
        if not points.empty:
            ax.scatter(
                points['long'], points['lat'], 
                c=color, 
                s=90, 
                edgecolors='black', 
                linewidth=0.8,
                label=label,
                zorder=score # Ensures higher severity points plot on top
            )
    
    ax.set_aspect('equal')
    ax.set_axis_off()
    ax.set_title(f"Ground Truth Infestation Distribution\n{field_name}", fontsize=18, fontweight='bold', pad=20)
    
    legend = ax.legend(title="Aleket Severity", loc='upper left', frameon=True, fontsize=12, title_fontsize=14)
    legend.get_frame().set_alpha(0.9)

    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{field_name}_single_map.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Generated single map: {output_path}")

long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    generate_single_infestation_map(file)

Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — EHI_tet_2012_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Geva_13_2012_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Kfar_horash_2011_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011_single_map.png
Generated single map: visualizations_infestation\Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011_single_map.png
Generated single map: visualizations_infestation\Boundary

# 📈 Temporal NDVI Trajectory Analysis
Visualizes the progression of physiological vigor (NDVI) over the 56-day observation window, segmented by final infestation severity. This provides a direct look into exactly *when* vegetative collapse begins in affected regions relative to healthy baselines.

In [9]:

def generate_temporal_trajectory(csv_path, output_dir="ndvi_field_trajectory"):
    os.makedirs(output_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    df = df.dropna(subset=['Days_Before_Sample', 'ndvi', 'Aleket_cnt'])
    
    # Ensure Aleket_cnt is an integer
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)
    
    # Strictly filter for Aleket scores 0 through 3
    df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
    
    if df.empty:
        print(f"    [SKIP] No data in 0-3 range for {field_name}")
        return None
    
    master_palette = {
        0: '#2CA02C', # Green
        1: '#FFD700', # Gold
        2: '#FF7F0E', # Orange
        3: '#D62728'  # Red
    }
            
    plt.figure(figsize=(10, 6))
    
    ax = sns.lineplot(
        data=df, 
        x='Days_Before_Sample', 
        y='ndvi', 
        hue='Aleket_cnt',
        palette=master_palette,
        marker='o',          
        markersize=8,
        linewidth=2.5,
        errorbar=('ci', 95)  
    )
    
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])
    
    plt.title(f"Temporal NDVI Trajectory by Infestation Severity\n{field_name}", 
              fontsize=16, fontweight='bold', pad=15)
    
    plt.xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    plt.ylabel("NDVI (Physiological Vigor)", fontsize=14)
    
    handles, labels = ax.get_legend_handles_labels()
    legend_labels = [f"Score {l}" for l in labels]
    plt.legend(handles=handles, labels=legend_labels, title="Aleket Severity", 
               title_fontsize=12, fontsize=11, loc='best', frameon=True, shadow=True)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{field_name}_ndvi_trajectory.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    return output_path

long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    print(f"Generating trajectory plot for: {file}")
    generate_temporal_trajectory(file)

Generating trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — EHI_tet_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
Generating trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Geva_13_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Geva_13_2012
Generating trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
Generating trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
Generating trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012_long_features.

# 🌀 Temporal Texture Dynamics (Green red ratio)


In [10]:


def generate_temporal_trajectory_gr(csv_path, output_dir="gr_ratio_field_trajectory"):
    os.makedirs(output_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    
    # Target green_red_ratio instead of ndvi
    df = df.dropna(subset=['Days_Before_Sample', 'green_red_ratio', 'Aleket_cnt'])
    
    # Ensure Aleket_cnt is an integer
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)
    
    # Strictly filter for Aleket scores 0 through 3
    df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
    
    if df.empty:
        print(f"    [SKIP] No data in 0-3 range for {field_name}")
        return None
    
    master_palette = {
        0: '#2CA02C', # Green
        1: '#FFD700', # Gold
        2: '#FF7F0E', # Orange
        3: '#D62728'  # Red
    }
            
    plt.figure(figsize=(10, 6))
    
    ax = sns.lineplot(
        data=df, 
        x='Days_Before_Sample', 
        y='green_red_ratio',  # Plotted metric
        hue='Aleket_cnt',
        palette=master_palette,
        marker='o',          
        markersize=8,
        linewidth=2.5,
        errorbar=('ci', 95)  
    )
    
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])
    
    plt.title(f"Temporal Green/Red Ratio Trajectory by Infestation Severity\n{field_name}", 
              fontsize=16, fontweight='bold', pad=15)
    
    plt.xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    plt.ylabel("Green/Red Ratio", fontsize=14)
    
    handles, labels = ax.get_legend_handles_labels()
    legend_labels = [f"Score {l}" for l in labels]
    plt.legend(handles=handles, labels=legend_labels, title="Aleket Severity", 
               title_fontsize=12, fontsize=11, loc='best', frameon=True, shadow=True)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{field_name}_gr_ratio_trajectory.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    return output_path

# Target the original directory, not 2025
long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    print(f"Generating GR Ratio trajectory plot for: {file}")
    generate_temporal_trajectory_gr(file)

Generating GR Ratio trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — EHI_tet_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
Generating GR Ratio trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Geva_13_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Geva_13_2012
Generating GR Ratio trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
Generating GR Ratio trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
Generating GR Ratio trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_201

# 📉 Temporal Texture Dynamics (Homogeneity)
This block generates the temporal trajectory of GLCM Homogeneity (structural similarity) across the observation window, helping track how structural uniformity changes per severity level.


In [11]:


def generate_temporal_trajectory_homo(csv_path, output_dir="homo_field_trajectory"):
    os.makedirs(output_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    
    # Target homogeneity instead of ndvi or gr_ratio
    df = df.dropna(subset=['Days_Before_Sample', 'homogeneity', 'Aleket_cnt'])
    
    # Ensure Aleket_cnt is an integer
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)
    
    # Strictly filter for Aleket scores 0 through 3
    df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
    
    if df.empty:
        print(f"    [SKIP] No data in 0-3 range for {field_name}")
        return None
    
    master_palette = {
        0: '#2CA02C', # Green
        1: '#FFD700', # Gold
        2: '#FF7F0E', # Orange
        3: '#D62728'  # Red
    }
            
    plt.figure(figsize=(10, 6))
    
    ax = sns.lineplot(
        data=df, 
        x='Days_Before_Sample', 
        y='homogeneity',  # Plotted metric
        hue='Aleket_cnt',
        palette=master_palette,
        marker='o',          
        markersize=8,
        linewidth=2.5,
        errorbar=('ci', 95)  
    )
    
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])
    
    plt.title(f"Temporal GLCM Homogeneity Trajectory by Infestation Severity\n{field_name}", 
              fontsize=16, fontweight='bold', pad=15)
    
    plt.xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    plt.ylabel("GLCM Homogeneity", fontsize=14)
    
    handles, labels = ax.get_legend_handles_labels()
    legend_labels = [f"Score {l}" for l in labels]
    plt.legend(handles=handles, labels=legend_labels, title="Aleket Severity", 
               title_fontsize=12, fontsize=11, loc='best', frameon=True, shadow=True)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{field_name}_homo_trajectory.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    return output_path

# Target the original directory
long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    print(f"Generating Homogeneity trajectory plot for: {file}")
    generate_temporal_trajectory_homo(file)

Generating Homogeneity trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — EHI_tet_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
Generating Homogeneity trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Geva_13_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Geva_13_2012
Generating Homogeneity trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
Generating Homogeneity trajectory plot for: field_temporal_datasets_long\Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011_long_features.csv
    [SKIP] No data in 0-3 range for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
Generating Homogeneity trajectory plot for: field_temporal_datasets_long\Boundary_Fiel

# 📊 Global Temporal Slopes & Full Correlation Matrix
Instead of single static observables, this phase calculates the explicit mathematical `slope` of temporal trajectories per point. It concludes by calculating global Spearman Rank correlation coefficients, outputting a high-level correlation heatmap to inform feature importance.

In [12]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_statistics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_full_correlation_matrix():
    files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR}")
        return

    features = [
        'ndvi', 'gndvi', 'green_red_ratio', 'ndre', 
        'entropy', 'green', 'homogeneity', 'asm'
    ]

    print(f"Processing {len(files)} field datasets...")
    summary_rows = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        for pt_id, group in df.groupby('Point_ID'):
            aleket = group['Aleket_cnt'].iloc[0]
            
            if pd.isna(aleket) or aleket not in [0, 1, 2, 3]:
                continue
                
            row = {
                'Field': field_name,
                'Point_ID': pt_id,
                'Aleket_cnt': aleket
            }
            
            # Convert Days_Before_Sample to a forward-moving time axis
            x_time = -1 * group['Days_Before_Sample'].values

            for feat in features:
                if feat not in group.columns:
                    row[f'{feat}_slope'] = np.nan
                    continue

                y_vals = group[feat].values
                
                valid_mask = ~np.isnan(y_vals) & ~np.isnan(x_time)
                x_valid = x_time[valid_mask]
                y_valid = y_vals[valid_mask]
                
                if len(x_valid) >= 2:
                    slope, intercept, r_value, p_value, std_err = linregress(x_valid, y_valid)
                    row[f'{feat}_slope'] = slope
                else:
                    row[f'{feat}_slope'] = np.nan
                    
            summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(f"\nExtracted temporal slopes for {len(summary_df)} total spatial points.")
    
    summary_csv_path = os.path.join(OUTPUT_DIR, "global_temporal_slopes.csv")
    summary_df.to_csv(summary_csv_path, index=False)
    
    # Compute Spearman Rank Correlation for the entire matrix
    corr_cols = ['Aleket_cnt'] + [f"{feat}_slope" for feat in features if f"{feat}_slope" in summary_df.columns]
    corr_matrix = summary_df[corr_cols].corr(method='spearman')

    # Generate Full Correlation Heatmap
    plt.figure(figsize=(14, 12))
    
    # Optional: Sort the matrix columns/rows by their correlation to Aleket_cnt for better readability
    sort_order = corr_matrix['Aleket_cnt'].sort_values(ascending=False).index
    corr_matrix_sorted = corr_matrix.loc[sort_order, sort_order]
    
    # Use a mask to hide the upper triangle (prevents redundant mirrored data)
    mask = np.triu(np.ones_like(corr_matrix_sorted, dtype=bool))
    
    sns.heatmap(
        corr_matrix_sorted, 
        mask=mask,
        annot=True, 
        cmap='coolwarm', 
        fmt=".2f", 
        vmin=-1, 
        vmax=1,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': 'Spearman Correlation Coefficient (ρ)', 'shrink': 0.8}
    )
    
    plt.title("Full Feature Correlation Matrix\n(Temporal Slopes & Infestation Severity)", fontsize=18, fontweight='bold', pad=20)
    
    # Rotate x-axis labels for readability
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(fontsize=11)
    plt.tight_layout()
    
    heatmap_path = os.path.join(OUTPUT_DIR, "global_full_correlation_matrix.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"\nSaved master slope table: {summary_csv_path}")
    print(f"Saved full correlation matrix heatmap: {heatmap_path}")

if __name__ == "__main__":
    build_full_correlation_matrix()

Processing 24 field datasets...

Extracted temporal slopes for 9357 total spatial points.

Saved master slope table: visualizations_statistics\global_temporal_slopes.csv
Saved full correlation matrix heatmap: visualizations_statistics\global_full_correlation_matrix.png


# ⏳ Time-Binned Correlation & Delta Analytics
Observation timelines are binned into generalized states: `Early`, `Mid`, and `Late`. By generating Delta Collapse metrics (e.g., Late vs Early states), we highlight extreme inflection points and correlate these binned transitions directly against ground-truth severity. This reveals how terminal phenomena compare to steady-states.

In [13]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_statistics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_binned_correlation_matrix():
    files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR}")
        return

    features = [
        'ndvi', 'gndvi', 'green_red_ratio', 'ndre', 
        'entropy', 'green', 'homogeneity', 'asm'
    ]

    print(f"Processing {len(files)} field datasets with Time Binning...")
    all_binned_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Drop rows missing sample dates or ground truth
        df = df.dropna(subset=['Days_Before_Sample', 'Aleket_cnt'])
        
        # 1. Define Temporal Bins
        # Late: 0-20 days | Mid: 21-40 days | Early: 41+ days
        bins = [-1, 20, 40, 150] 
        labels = ['Late_0_20', 'Mid_21_40', 'Early_41_plus']
        df['Time_Window'] = pd.cut(df['Days_Before_Sample'], bins=bins, labels=labels)
        
        # Filter out invalid Aleket scores
        df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
        df['Field'] = field_name
        
        # 2. Aggregate Data by Bin (Using Median to resist atmospheric outliers)
        grouped = df.groupby(['Field', 'Point_ID', 'Aleket_cnt', 'Time_Window'], observed=True)[features].median().reset_index()
        all_binned_data.append(grouped)

    if not all_binned_data:
        print("[ERROR] No valid data extracted.")
        return

    # Combine all fields into a master dataframe
    master_df = pd.concat(all_binned_data, ignore_index=True)

    # 3. Pivot the Table (Wide Format)
    # Every Point_ID gets exactly one row, with columns like 'ndvi_Late_0_20', 'ndvi_Early_41_plus'
    pivot_df = master_df.pivot(index=['Field', 'Point_ID', 'Aleket_cnt'], 
                               columns='Time_Window', 
                               values=features).reset_index()
    
    # Flatten the MultiIndex columns created by the pivot
    pivot_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in pivot_df.columns]

    # 4. Calculate the Delta (The "Collapse" Feature)
    # Delta = Final State (Late) - Initial State (Early)
    # A highly negative NDVI Delta means a massive crash. A highly positive Entropy Delta means a massive spike in chaos.
    delta_cols = []
    for feat in features:
        early_col = f"{feat}_Early_41_plus"
        late_col = f"{feat}_Late_0_20"
        
        if early_col in pivot_df.columns and late_col in pivot_df.columns:
            delta_name = f"{feat}_Delta_Collapse"
            pivot_df[delta_name] = pivot_df[late_col] - pivot_df[early_col]
            delta_cols.append(delta_name)

    print(f"\nExtracted binned features for {len(pivot_df)} total spatial points.")
    
    # Save the master table for ML Training
    summary_csv_path = os.path.join(OUTPUT_DIR, "global_binned_features.csv")
    pivot_df.to_csv(summary_csv_path, index=False)
    
    # 5. Compute Spearman Rank Correlation
    # We correlate Aleket_cnt against the new Delta metrics and the Final State (Late) metrics
    late_cols = [f"{feat}_Late_0_20" for feat in features if f"{feat}_Late_0_20" in pivot_df.columns]
    target_columns = ['Aleket_cnt'] + delta_cols + late_cols
    
    corr_matrix = pivot_df[target_columns].corr(method='spearman')

    # 6. Generate Correlation Heatmap
    plt.figure(figsize=(16, 14))
    
    sort_order = corr_matrix['Aleket_cnt'].sort_values(ascending=False).index
    corr_matrix_sorted = corr_matrix.loc[sort_order, sort_order]
    
    mask = np.triu(np.ones_like(corr_matrix_sorted, dtype=bool))
    
    sns.heatmap(
        corr_matrix_sorted, 
        mask=mask,
        annot=True, 
        cmap='coolwarm', 
        fmt=".2f", 
        vmin=-1, 
        vmax=1,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': 'Spearman Correlation Coefficient (ρ)', 'shrink': 0.8}
    )
    
    plt.title("Time-Binned Feature Correlation Matrix\n(Deltas & Late-Stage Variables vs. Severity)", fontsize=18, fontweight='bold', pad=20)
    
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.tight_layout()
    
    heatmap_path = os.path.join(OUTPUT_DIR, "global_binned_correlation_matrix.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # Print the specific correlations to Aleket_cnt to the console
    print("\n--- Spearman Correlation with Aleket_cnt ---")
    print(corr_matrix['Aleket_cnt'].sort_values(ascending=False).to_string())
    print(f"\nSaved master binned table: {summary_csv_path}")
    print(f"Saved binned correlation matrix: {heatmap_path}")

if __name__ == "__main__":
    build_binned_correlation_matrix()

Processing 24 field datasets with Time Binning...

Extracted binned features for 9472 total spatial points.

--- Spearman Correlation with Aleket_cnt ---
Aleket_cnt                        1.0
ndvi_Delta_Collapse               NaN
gndvi_Delta_Collapse              NaN
green_red_ratio_Delta_Collapse    NaN
ndre_Delta_Collapse               NaN
entropy_Delta_Collapse            NaN
green_Delta_Collapse              NaN
homogeneity_Delta_Collapse        NaN
asm_Delta_Collapse                NaN
ndvi_Late_0_20                    NaN
gndvi_Late_0_20                   NaN
green_red_ratio_Late_0_20         NaN
ndre_Late_0_20                    NaN
entropy_Late_0_20                 NaN
green_Late_0_20                   NaN
homogeneity_Late_0_20             NaN
asm_Late_0_20                     NaN

Saved master binned table: visualizations_statistics\global_binned_features.csv
Saved binned correlation matrix: visualizations_statistics\global_binned_correlation_matrix.png


# ✨ Multivariate Signature Migration
A fascinating faceted scatter matrix observing the combined state of `NDVI` (Physiology) and `Entropy` (Structure) mapped across the temporal axes. It visually portrays how distinct severity clusters migrate dynamically across multi-dimensional space approaching the collapse date.

In [14]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_scatter_signatures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_multivariate_scatter(csv_path):
    """
    Generates faceted scatter plots (NDVI vs Entropy) tracking the 
    migration of healthy vs infected points over time.
    """
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    
    # Clean data (ensure we have values for the axes, time, and severity)
    df = df.dropna(subset=['ndvi', 'entropy', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty:
        print(f"[SKIP] No valid data for {field_name}")
        return
        
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)
    
    # Custom palette matching previous charts
    master_palette = {
        0: '#2CA02C', # Green (Healthy)
        1: '#FFD700', # Gold (Low)
        2: '#FF7F0E', # Orange (Medium)
        3: '#D62728'  # Red (High Severity)
    }
    
    # Dynamically handle unexpected classes (fallback to grey)
    available_scores = df['Aleket_cnt'].unique()
    custom_palette = {score: master_palette.get(score, '#7F7F7F') for score in available_scores}
    
    # Sort time steps chronologically (e.g., 50 days -> 10 days -> 0 days)
    time_steps = sorted(df['Days_Before_Sample'].unique(), reverse=True)
    n_steps = len(time_steps)
    
    if n_steps == 0:
        return
        
    # Setup faceted figure
    fig, axes = plt.subplots(1, n_steps, figsize=(6 * n_steps, 6), squeeze=False)
    axes = axes.flatten()
    
    # --- CRITICAL: Set Global Axis Limits ---
    # To accurately visualize the points "migrating" across the space over time,
    # the X and Y axes must be identical for every facet subplot.
    x_min, x_max = df['ndvi'].min() - 0.05, df['ndvi'].max() + 0.05
    y_min, y_max = df['entropy'].min() - 0.2, df['entropy'].max() + 0.2
    
    for i, days in enumerate(time_steps):
        ax = axes[i]
        subset = df[df['Days_Before_Sample'] == days]
        
        if not subset.empty:
            # Generate the scatter plot
            sns.scatterplot(
                data=subset, 
                x='ndvi', 
                y='entropy', 
                hue='Aleket_cnt',
                palette=custom_palette, 
                ax=ax, 
                s=90,             # Marker size
                edgecolor='black',# High-contrast borders
                alpha=0.85        # Slight transparency to handle overlaps
            )
        
        # Apply global limits and labels
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        
        ax.set_title(f"{days} Days Before Sample", fontsize=15, fontweight='bold', pad=10)
        ax.set_xlabel("NDVI (Physiological Vigor)", fontsize=13)
        
        # Only put Y-axis labels and the Legend on the very first subplot 
        # to avoid redundancy and save visual space
        if i == 0:
            ax.set_ylabel("GLCM Entropy (Structural Chaos)", fontsize=13)
            ax.legend(title="Aleket Severity", loc='best', framealpha=0.9, fontsize=11, title_fontsize=12)
        else:
            ax.set_ylabel("")
            if ax.get_legend():
                ax.get_legend().remove()
                
    # Add a global title
    plt.suptitle(f"Multivariate Signature Migration (NDVI vs Entropy): {field_name}", fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()
    
    # Save the output
    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_scatter_signature.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"Generated Multivariate Signature: {out_path}")

# Run execution for all files in the directory
long_files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))

if not long_files:
    # Fallback to current directory for testing
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    generate_multivariate_scatter(file)

[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Geva_13_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_horash_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_um_savisa_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_hamra_vav
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_shokef_alef
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_kemach_E
[SKIP] No valid data for Boun

## ndre vs green_red_ratio in time

# 🧪 Biochemical Signature Scatter Plots
This section generates faceted scatter plots for each field, tracking the relationship between the Green/Red Ratio (chlorosis) and NDRE (early stress/chlorophyll). The visualization maps the migration of the crop's biochemical signature over the time steps leading to the sample date.

In [15]:

# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_scatter_biochemical"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_biochemical_scatter(csv_path):
    """
    Generates faceted scatter plots (Green/Red Ratio vs NDRE) tracking the 
    biochemical/pigment degradation of the crop over time.
    """
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")
    
    # Clean data for the specific features
    df = df.dropna(subset=['ndre', 'green_red_ratio', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty:
        print(f"[SKIP] No valid data for {field_name}")
        return
        
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)
    
    master_palette = {
        0: '#2CA02C', # Green
        1: '#FFD700', # Gold
        2: '#FF7F0E', # Orange
        3: '#D62728'  # Red
    }
    
    available_scores = df['Aleket_cnt'].unique()
    custom_palette = {score: master_palette.get(score, '#7F7F7F') for score in available_scores}
    
    time_steps = sorted(df['Days_Before_Sample'].unique(), reverse=True)
    n_steps = len(time_steps)
    
    if n_steps == 0:
        return
        
    fig, axes = plt.subplots(1, n_steps, figsize=(6 * n_steps, 6), squeeze=False)
    axes = axes.flatten()
    
    # --- CRITICAL: Set Global Axis Limits for these specific metrics ---
    x_min, x_max = df['green_red_ratio'].min() - 0.1, df['green_red_ratio'].max() + 0.1
    y_min, y_max = df['ndre'].min() - 0.05, df['ndre'].max() + 0.05
    
    for i, days in enumerate(time_steps):
        ax = axes[i]
        subset = df[df['Days_Before_Sample'] == days]
        
        if not subset.empty:
            sns.scatterplot(
                data=subset, 
                x='green_red_ratio', 
                y='ndre', 
                hue='Aleket_cnt',
                palette=custom_palette, 
                ax=ax, 
                s=90,             
                edgecolor='black',
                alpha=0.85        
            )
        
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        
        ax.set_title(f"{days} Days Before Sample", fontsize=15, fontweight='bold', pad=10)
        ax.set_xlabel("Green/Red Ratio (Chlorosis Shift)", fontsize=13)
        
        if i == 0:
            ax.set_ylabel("NDRE (Chlorophyll/Early Stress)", fontsize=13)
            ax.legend(title="Aleket Severity", loc='best', framealpha=0.9, fontsize=11, title_fontsize=12)
        else:
            ax.set_ylabel("")
            if ax.get_legend():
                ax.get_legend().remove()
                
    plt.suptitle(f"Biochemical Signature Migration: Chlorosis vs NDRE\n{field_name}", fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()
    
    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_biochemical_signature.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"Generated Biochemical Signature: {out_path}")

# Run execution
long_files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for file in long_files:
    generate_biochemical_scatter(file)

[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Geva_13_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_horash_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_um_savisa_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_hamra_vav
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_shokef_alef
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_kemach_E
[SKIP] No valid data for Boun

# 🗺️ Grid-Based Temporal Trajectory
This block overlays a 30x30m quadrat grid over the field boundaries to assign patch-level severity scores. It then plots the aggregated NDVI trajectory of these patches over time, revealing how different infestation severity classes evolve chronologically.

In [16]:

# --- Configuration ---
OUTPUT_DIR = "visualizations_grid_trajectory"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Metric projection (UTM Zone 36N) to convert GPS Degrees to physical Meters
TARGET_CRS = "EPSG:32636" 
GRID_SIZE_M = 30 # Using 30x30m quadrats to capture a regional patch

def generate_grid_temporal_trajectory(csv_path):
    """
    Overlays a 30x30m grid, assigns a patch-level severity score, 
    and plots the aggregated NDVI trajectory of the patches over time.
    """
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")

    # Clean data
    df = df.dropna(subset=['lat', 'long', 'ndvi', 'entropy', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty:
        print(f"[SKIP] No valid data for {field_name}")
        return None

    # Convert to GeoDataFrame and project to metric units
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['long'], df['lat']),
        crs="EPSG:4326"
    )
    gdf_metric = gdf.to_crs(TARGET_CRS)

    # Generate Bounding Box limits based on the static GPS points
    unique_pts = gdf_metric.drop_duplicates(subset=['Point_ID'])
    xmin, ymin, xmax, ymax = unique_pts.total_bounds

    # Create the regular quadrat grid covering the field bounds
    grid_cells = []
    for x in np.arange(xmin, xmax, GRID_SIZE_M):
        for y in np.arange(ymin, ymax, GRID_SIZE_M):
            grid_cells.append(box(x, y, x + GRID_SIZE_M, y + GRID_SIZE_M))

    grid = gpd.GeoDataFrame({'grid_id': range(len(grid_cells))}, geometry=grid_cells, crs=TARGET_CRS)

    # Perform Spatial Join (assign each GPS point to its respective 30m grid cell)
    joined = gpd.sjoin(gdf_metric, grid, how="inner", predicate="within")

    # 1. Determine Grid Severity 
    # Calculate the Mean Aleket score for the entire 30m patch
    grid_severity = joined.drop_duplicates(subset=['Point_ID']).groupby('grid_id')['Aleket_cnt'].mean().reset_index()

    # Categorize the numeric mean into distinct severity classes for the line chart
    def classify_severity(val):
        if val == 0: return 'Healthy Patch (0)'
        elif val <= 1.0: return 'Low Infestation (0.1-1.0)'
        elif val <= 2.0: return 'Medium Infestation (1.1-2.0)'
        else: return 'High Infestation (>2.0)'
        
    grid_severity['Grid_Class'] = grid_severity['Aleket_cnt'].apply(classify_severity)

    # 2. Aggregate Temporal Data per Grid Cell
    # Median is used to resist extreme outlier pixels caused by shadows or soil
    temporal_agg = joined.groupby(['grid_id', 'Days_Before_Sample']).agg({
        'ndvi': 'median',
        'entropy': 'median',
        'Point_ID': 'nunique' 
    }).reset_index()

    # Filter out grids that captured zero or isolated single points (adjust threshold as needed based on sampling density)
    temporal_agg = temporal_agg[temporal_agg['Point_ID'] >= 1] 

    # Merge the severity class back to the aggregated temporal data
    final_df = pd.merge(temporal_agg, grid_severity[['grid_id', 'Grid_Class']], on='grid_id')

    if final_df.empty:
        return None

    # Consistent Color Palette mapping
    palette = {
        'Healthy Patch (0)': '#2CA02C',
        'Low Infestation (0.1-1.0)': '#FFD700',
        'Medium Infestation (1.1-2.0)': '#FF7F0E',
        'High Infestation (>2.0)': '#D62728'
    }
    
    # Filter palette to only include classes actually present in the data
    custom_palette = {k: v for k, v in palette.items() if k in final_df['Grid_Class'].unique()}

    # --- Plotting the Trajectory ---
    plt.figure(figsize=(10, 6))
    
    # Seaborn lineplot averages all grids within the same severity class and adds a 95% Confidence Interval
    ax = sns.lineplot(
        data=final_df,
        x='Days_Before_Sample',
        y='ndvi',
        hue='Grid_Class',
        palette=custom_palette,
        marker='o',
        markersize=8,
        linewidth=2.5,
        errorbar=('ci', 95)
    )

    # Reverse X axis so time flows left to right
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])

    # Academic Labels
    plt.title(f"Grid-Aggregated Temporal NDVI Trajectory\n{field_name} ({GRID_SIZE_M}x{GRID_SIZE_M}m Patches)", 
              fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    plt.ylabel("Aggregated Patch NDVI (Median)", fontsize=14)

    plt.legend(title="Patch Infestation Level", title_fontsize=12, fontsize=11, loc='best', frameon=True, shadow=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Save output
    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_grid_ndvi_trajectory.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"Generated Grid Trajectory: {out_path}")
    return out_path

# Execute across available datasets
long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
if not long_files:
    long_files = glob.glob("*_long_features.csv")
    
for f in long_files:
    generate_grid_temporal_trajectory(f)

[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — EHI_tet_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Geva_13_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Havat_Gadash_2_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_b_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_blum_rekiak_E_2012
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Kfar_horash_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Maale_gilboa_makruz_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_mafrek_E_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Mesilot_um_savisa_2011
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_hamra_vav
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_2011_shokef_alef
[SKIP] No valid data for Boundary_Fields_Addition_2010-12 — Sde_Eliyahu_kemach_E
[SKIP] No valid data for Boun

# 🎯 Temporal Buffer Analysis
This cell analyzes the correlation between the distance from the field border and the temporal degradation of vegetation indices (GNDVI and NDRE). It generates a scatter plot overlaid with a regression line to investigate whether edge effects impact crop health over time.

In [17]:
# --- Configuration ---
OUTPUT_DIR = "visualizations_temporal_buffer"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_CRS = "EPSG:32636"

def generate_temporal_buffer_analysis(csv_path):
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")

    # Clean data - Using GNDVI and NDRE as requested
    df = df.dropna(subset=['lat', 'long', 'gndvi', 'ndre', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty: return None

    # Ensure Aleket_cnt is int to match dict keys consistently
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int)

    # 1. Calculate Temporal Slopes per Point_ID
    df['Time_Forward'] = -1 * df['Days_Before_Sample']
    
    slopes = []
    for pt_id, group in df.groupby('Point_ID'):
        aleket = group['Aleket_cnt'].iloc[0]
        lat = group['lat'].iloc[0]
        lon = group['long'].iloc[0]
        
        row = {'Point_ID': pt_id, 'Aleket_cnt': aleket, 'lat': lat, 'long': lon}
        
        if len(group) >= 2:
            # GNDVI Slope
            valid_gndvi = group.dropna(subset=['gndvi', 'Time_Forward'])
            if len(valid_gndvi) >= 2:
                m, _, _, _, _ = linregress(valid_gndvi['Time_Forward'], valid_gndvi['gndvi'])
                row['gndvi_slope'] = m
            else:
                row['gndvi_slope'] = np.nan
                
            # NDRE Slope
            valid_ndre = group.dropna(subset=['ndre', 'Time_Forward'])
            if len(valid_ndre) >= 2:
                m, _, _, _, _ = linregress(valid_ndre['Time_Forward'], valid_ndre['ndre'])
                row['ndre_slope'] = m
            else:
                row['ndre_slope'] = np.nan
        else:
            row['gndvi_slope'] = np.nan
            row['ndre_slope'] = np.nan
            
        slopes.append(row)

    slope_df = pd.DataFrame(slopes).dropna(subset=['gndvi_slope', 'ndre_slope'])
    if slope_df.empty: return None

    # 2. Convert to GeoDataFrame and compute Distance to Border
    gdf = gpd.GeoDataFrame(
        slope_df, geometry=gpd.points_from_xy(slope_df['long'], slope_df['lat']), crs="EPSG:4326"
    ).to_crs(TARGET_CRS)
    
    convex_hull = gdf.geometry.unary_union.convex_hull
    field_boundary = convex_hull.boundary 
    gdf['dist_to_border'] = gdf.geometry.distance(field_boundary)

    # --- FIX: Dynamic Palette ---
    master_palette = {0: '#2CA02C', 1: '#FFD700', 2: '#FF7F0E', 3: '#D62728'}
    available_scores = gdf['Aleket_cnt'].unique()
    custom_palette = {}
    for score in available_scores:
        if score in master_palette:
            custom_palette[score] = master_palette[score]
        else:
            custom_palette[score] = '#7F7F7F' # Assign grey for outlier classes like '7'

    # 3. Visualization: Scatter plot of Distance vs Slope
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Distance vs GNDVI Slope
    # Use scatterplot to handle Hue correctly, then overlay regplot without scatter
    sns.scatterplot(data=gdf, x='dist_to_border', y='gndvi_slope', hue='Aleket_cnt', 
                    palette=custom_palette, ax=axes[0], s=60, edgecolor='black', alpha=0.8)
    sns.regplot(data=gdf, x='dist_to_border', y='gndvi_slope', ax=axes[0], scatter=False, 
                line_kws={'color':'#1f77b4', 'linestyle':'--', 'linewidth':2})
    
    axes[0].set_title("Distance from Border vs. GNDVI Degradation Rate", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Distance from Field Border (Meters)", fontsize=12)
    axes[0].set_ylabel("GNDVI Slope (Rate of Change)", fontsize=12)
    axes[0].grid(True, linestyle=':', alpha=0.6)
    axes[0].legend(title="Aleket Score")

    # Plot 2: Distance vs NDRE Slope
    sns.scatterplot(data=gdf, x='dist_to_border', y='ndre_slope', hue='Aleket_cnt', 
                    palette=custom_palette, ax=axes[1], s=60, edgecolor='black', alpha=0.8)
    sns.regplot(data=gdf, x='dist_to_border', y='ndre_slope', ax=axes[1], scatter=False, 
                line_kws={'color':'#1f77b4', 'linestyle':'--', 'linewidth':2})
    
    axes[1].set_title("Distance from Border vs. NDRE Degradation Rate", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Distance from Field Border (Meters)", fontsize=12)
    axes[1].set_ylabel("NDRE Slope (Rate of Change)", fontsize=12)
    axes[1].grid(True, linestyle=':', alpha=0.6)
    axes[1].legend(title="Aleket Score")

    plt.suptitle(f"Temporal Gradient Analysis (Distance to Border)\n{field_name}", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_temporal_buffer.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    return out_path

long_files = glob.glob("field_temporal_datasets_long/*_long_features.csv")
for f in long_files:
    generate_temporal_buffer_analysis(f)

# 🦠 Aleket Comparison (Green/Red Ratio & ASM)
This block statistically compares the degradation rates of healthy versus infected patches. By analyzing the slopes of the Green/Red Ratio (biochemical trait) and GLCM ASM (textural trait), it visualizes the temporal decay differences via box plots and strip plots, accompanied by Mann-Whitney U significance testing.

In [18]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_aleket_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_aleket_boxplot(csv_path):
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")

    df = df.dropna(subset=['green_red_ratio', 'asm', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty: return None

    # Filter strictly for valid ground truth labels (0 to 3)
    df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
    df['Time_Forward'] = -1 * df['Days_Before_Sample']
    
    slopes = []
    for pt_id, group in df.groupby('Point_ID'):
        aleket = group['Aleket_cnt'].iloc[0]
        row = {'Point_ID': pt_id, 'Aleket_cnt': aleket}
        
        # Explicit classification based strictly on Aleket values
        row['Infestation_Status'] = 'Aleket > 0\n(Infested)' if aleket > 0 else 'Aleket = 0\n(Healthy)'
        
        if len(group) >= 2:
            # Biochemical Slope (Green/Red Ratio)
            valid_gr = group.dropna(subset=['green_red_ratio', 'Time_Forward'])
            if len(valid_gr) >= 2:
                m, _, _, _, _ = linregress(valid_gr['Time_Forward'], valid_gr['green_red_ratio'])
                row['gr_ratio_slope'] = m
            else:
                row['gr_ratio_slope'] = np.nan
                
            # Textural Slope (GLCM ASM)
            valid_asm = group.dropna(subset=['asm', 'Time_Forward'])
            if len(valid_asm) >= 2:
                m, _, _, _, _ = linregress(valid_asm['Time_Forward'], valid_asm['asm'])
                row['asm_slope'] = m
            else:
                row['asm_slope'] = np.nan
        else:
            row['gr_ratio_slope'] = np.nan
            row['asm_slope'] = np.nan
            
        slopes.append(row)

    slope_df = pd.DataFrame(slopes).dropna(subset=['gr_ratio_slope', 'asm_slope'])
    
    if slope_df.empty or len(slope_df['Infestation_Status'].unique()) < 2:
        print(f"[SKIP] Requires both Aleket > 0 and Aleket = 0 points for comparison: {field_name}")
        return None

    # --- Statistical Testing (Mann-Whitney U) ---
    infected = slope_df[slope_df['Infestation_Status'] == 'Aleket > 0\n(Infested)']
    healthy = slope_df[slope_df['Infestation_Status'] == 'Aleket = 0\n(Healthy)']
    
    stat_gr, p_gr = mannwhitneyu(infected['gr_ratio_slope'], healthy['gr_ratio_slope'], alternative='two-sided')
    stat_asm, p_asm = mannwhitneyu(infected['asm_slope'], healthy['asm_slope'], alternative='two-sided')

    # --- Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    palette = {'Aleket > 0\n(Infested)': '#D62728', 'Aleket = 0\n(Healthy)': '#2CA02C'}
    
    # Plot 1: Green/Red Ratio Slope
    sns.boxplot(data=slope_df, x='Infestation_Status', y='gr_ratio_slope', palette=palette, ax=axes[0], 
                width=0.5, boxprops=dict(alpha=0.8), fliersize=5)
    
    sns.stripplot(data=slope_df, x='Infestation_Status', y='gr_ratio_slope', color='black', alpha=0.4, ax=axes[0], jitter=True)
    
    axes[0].set_title("Biochemical Degradation\n(Green/Red Ratio Slope)", fontsize=14, fontweight='bold')
    axes[0].set_ylabel("Rate of Change (Slope)", fontsize=12)
    axes[0].set_xlabel("")
    axes[0].grid(True, axis='y', linestyle=':', alpha=0.7)
    
    significance_gr = f"p = {p_gr:.4f}" + (" *" if p_gr < 0.05 else " (ns)")
    axes[0].text(0.5, 0.95, significance_gr, ha='center', va='center', transform=axes[0].transAxes, 
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))

    # Plot 2: GLCM ASM Slope
    sns.boxplot(data=slope_df, x='Infestation_Status', y='asm_slope', palette=palette, ax=axes[1], 
                width=0.5, boxprops=dict(alpha=0.8), fliersize=5)
    
    sns.stripplot(data=slope_df, x='Infestation_Status', y='asm_slope', color='black', alpha=0.4, ax=axes[1], jitter=True)
    
    axes[1].set_title("Structural Degradation\n(GLCM ASM Slope)", fontsize=14, fontweight='bold')
    axes[1].set_ylabel("Rate of Change (Slope)", fontsize=12)
    axes[1].set_xlabel("")
    axes[1].grid(True, axis='y', linestyle=':', alpha=0.7)
    
    significance_asm = f"p = {p_asm:.4f}" + (" *" if p_asm < 0.05 else " (ns)")
    axes[1].text(0.5, 0.95, significance_asm, ha='center', va='center', transform=axes[1].transAxes, 
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))

    plt.suptitle(f"Temporal Decay: Aleket > 0 vs. Aleket = 0\n{field_name}", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_aleket_comparison.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Aleket Comparison: {out_path}")
    return out_path

long_files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for f in long_files:
    generate_aleket_boxplot(f)

[SKIP] Requires both Aleket > 0 and Aleket = 0 points for comparison: Boundary_Fields_Baseline_2011 — Hulata_krad_3_4


# 📊 Aleket Comparison (NDVI & Homogeneity)
This step performs a statistical comparison between healthy (Aleket = 0) and infected (Aleket > 0) patches. It evaluates the degradation slopes of physiological (NDVI) and structural (GLCM Homogeneity) traits over time, using the Mann-Whitney U test to assess statistical significance.

In [19]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_aleket_comparison_2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_aleket_boxplot_ndvi_homo(csv_path):
    df = pd.read_csv(csv_path)
    field_name = os.path.basename(csv_path).replace("_long_features.csv", "")

    # Clean data - substituted green_red_ratio and asm with ndvi and homogeneity
    df = df.dropna(subset=['ndvi', 'homogeneity', 'Aleket_cnt', 'Days_Before_Sample'])
    if df.empty: return None

    # Filter strictly for valid ground truth labels (0 to 3)
    df = df[df['Aleket_cnt'].isin([0, 1, 2, 3])]
    df['Time_Forward'] = -1 * df['Days_Before_Sample']
    
    slopes = []
    for pt_id, group in df.groupby('Point_ID'):
        aleket = group['Aleket_cnt'].iloc[0]
        row = {'Point_ID': pt_id, 'Aleket_cnt': aleket}
        
        # Explicit classification based strictly on Aleket values
        row['Infestation_Status'] = 'Aleket > 0\n(Infested)' if aleket > 0 else 'Aleket = 0\n(Healthy)'
        
        if len(group) >= 2:
            # Physiological Slope (NDVI)
            valid_ndvi = group.dropna(subset=['ndvi', 'Time_Forward'])
            if len(valid_ndvi) >= 2:
                m, _, _, _, _ = linregress(valid_ndvi['Time_Forward'], valid_ndvi['ndvi'])
                row['ndvi_slope'] = m
            else:
                row['ndvi_slope'] = np.nan
                
            # Textural Slope (GLCM Homogeneity)
            valid_homo = group.dropna(subset=['homogeneity', 'Time_Forward'])
            if len(valid_homo) >= 2:
                m, _, _, _, _ = linregress(valid_homo['Time_Forward'], valid_homo['homogeneity'])
                row['homogeneity_slope'] = m
            else:
                row['homogeneity_slope'] = np.nan
        else:
            row['ndvi_slope'] = np.nan
            row['homogeneity_slope'] = np.nan
            
        slopes.append(row)

    slope_df = pd.DataFrame(slopes).dropna(subset=['ndvi_slope', 'homogeneity_slope'])
    
    if slope_df.empty or len(slope_df['Infestation_Status'].unique()) < 2:
        print(f"[SKIP] Requires both Aleket > 0 and Aleket = 0 points for comparison: {field_name}")
        return None

    # --- Statistical Testing (Mann-Whitney U) ---
    infected = slope_df[slope_df['Infestation_Status'] == 'Aleket > 0\n(Infested)']
    healthy = slope_df[slope_df['Infestation_Status'] == 'Aleket = 0\n(Healthy)']
    
    stat_ndvi, p_ndvi = mannwhitneyu(infected['ndvi_slope'], healthy['ndvi_slope'], alternative='two-sided')
    stat_homo, p_homo = mannwhitneyu(infected['homogeneity_slope'], healthy['homogeneity_slope'], alternative='two-sided')

    # --- Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    palette = {'Aleket > 0\n(Infested)': '#D62728', 'Aleket = 0\n(Healthy)': '#2CA02C'}
    
    # Plot 1: NDVI Slope
    sns.boxplot(data=slope_df, x='Infestation_Status', y='ndvi_slope', palette=palette, ax=axes[0], 
                width=0.5, boxprops=dict(alpha=0.8), fliersize=5)
    
    sns.stripplot(data=slope_df, x='Infestation_Status', y='ndvi_slope', color='black', alpha=0.4, ax=axes[0], jitter=True)
    
    axes[0].set_title("Physiological Degradation\n(NDVI Slope)", fontsize=14, fontweight='bold')
    axes[0].set_ylabel("Rate of Change (Slope)", fontsize=12)
    axes[0].set_xlabel("")
    axes[0].grid(True, axis='y', linestyle=':', alpha=0.7)
    
    significance_ndvi = f"p = {p_ndvi:.4f}" + (" *" if p_ndvi < 0.05 else " (ns)")
    axes[0].text(0.5, 0.95, significance_ndvi, ha='center', va='center', transform=axes[0].transAxes, 
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))

    # Plot 2: GLCM Homogeneity Slope
    sns.boxplot(data=slope_df, x='Infestation_Status', y='homogeneity_slope', palette=palette, ax=axes[1], 
                width=0.5, boxprops=dict(alpha=0.8), fliersize=5)
    
    sns.stripplot(data=slope_df, x='Infestation_Status', y='homogeneity_slope', color='black', alpha=0.4, ax=axes[1], jitter=True)
    
    axes[1].set_title("Structural Degradation\n(GLCM Homogeneity Slope)", fontsize=14, fontweight='bold')
    axes[1].set_ylabel("Rate of Change (Slope)", fontsize=12)
    axes[1].set_xlabel("")
    axes[1].grid(True, axis='y', linestyle=':', alpha=0.7)
    
    significance_homo = f"p = {p_homo:.4f}" + (" *" if p_homo < 0.05 else " (ns)")
    axes[1].text(0.5, 0.95, significance_homo, ha='center', va='center', transform=axes[1].transAxes, 
                 fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))

    plt.suptitle(f"Temporal Decay: Aleket > 0 vs. Aleket = 0\n{field_name}", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"{field_name}_aleket_comparison_ndvi_homo.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Aleket Comparison: {out_path}")
    return out_path

long_files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
if not long_files:
    long_files = glob.glob("*_long_features.csv")

for f in long_files:
    generate_aleket_boxplot_ndvi_homo(f)

[SKIP] Requires both Aleket > 0 and Aleket = 0 points for comparison: Boundary_Fields_Baseline_2011 — Hulata_krad_3_4


# 🌐 Global Trajectory Trends (NDVI & Entropy)
Here, we aggregate the temporal data across all monitored fields. It creates a 'spaghetti plot' showing individual field trajectories alongside a LOESS-smoothed global trend line, distinguishing between healthy and infected populations for NDVI and GLCM Entropy metrics.

In [20]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_global_trends"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_plot():
    files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
    if not files:
        # Fallback to local dir for testing
        files = glob.glob("*_long_features.csv")
        
    if not files:
        print(f"[ERROR] No files found.")
        return

    print(f"Aggregating data from {len(files)} fields...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Clean crucial columns
        df = df.dropna(subset=['ndvi', 'entropy', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid data extracted.")
        return

    # 1. Build Master Global Dataset
    master_df = pd.concat(all_data, ignore_index=True)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3])]

    # 2. Binary Classification (Healthy vs Infected)
    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    # 3. Aggregate Data (Average per Field, per Status, per Time Step)
    # This prevents fields with 1,000 points from overwhelming fields with 100 points
    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'ndvi': 'mean',
        'entropy': 'mean'
    }).reset_index()

    # --- Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    metrics = [
        {'col': 'ndvi', 'ax_idx': 0, 'title': 'Global Vigor Trajectory (NDVI)', 'ylabel': 'Mean NDVI per Field'},
        {'col': 'entropy', 'ax_idx': 1, 'title': 'Global Structural Trajectory (GLCM Entropy)', 'ylabel': 'Mean Entropy per Field'}
    ]

    for m in metrics:
        ax = axes[m['ax_idx']]
        
        for status, color in palette.items():
            subset = agg_df[(agg_df['Status'] == status) & (agg_df[m['col']].notna())]
            
            if subset.empty:
                continue

            # A. Plot the Spaghetti (Individual Field Trajectories)
            sns.lineplot(
                data=subset, 
                x='Days_Before_Sample', 
                y=m['col'],
                units='Field',      # Group lines by field
                estimator=None,     # Do not average here; plot each unit
                color=color, 
                alpha=0.15,         # Faint transparency
                linewidth=1.5, 
                ax=ax
            )

            # B. Plot the LOESS Trendline (Global Smoothed Trend)
            sns.regplot(
                data=subset, 
                x='Days_Before_Sample', 
                y=m['col'],
                scatter=False,      # Hide the raw dots
                lowess=True,        # Apply Locally Estimated Scatterplot Smoothing
                color=color,
                line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
                ax=ax
            )

        # Formatting
        ax.set_title(m['title'], fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=13)
        ax.set_ylabel(m['ylabel'], fontsize=13)
        ax.grid(True, linestyle=':', alpha=0.7)

        # Reverse X-axis so time flows left to right (e.g., 50 days down to 0 days)
        current_xlim = ax.get_xlim()
        if current_xlim[0] < current_xlim[1]:
            ax.set_xlim(current_xlim[1], current_xlim[0])

    # 4. Custom Legend
    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    axes[0].legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend', 'Individual Field Trajectory'], 
        loc='best', fontsize=11, framealpha=0.9
    )
    
    # Remove second legend to keep it clean
    if axes[1].get_legend(): axes[1].get_legend().remove()

    plt.suptitle("Global Spatio-Temporal Analysis (LOESS Smoothing)", fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_trends.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_plot()

Aggregating data from 24 fields...
Generated Global Spaghetti Plot: visualizations_global_trends\global_spaghetti_loess_trends.png


# 📈 Global Trajectory Trends (Green/Red Ratio & Homogeneity)
Similar to the previous global analysis, this cell visualizes the aggregated temporal trajectories across all fields. However, it focuses on the biochemical shift (Green/Red Ratio) and structural homogeneity (GLCM Homogeneity), plotting both individual field data and LOESS-smoothed global trends.

In [21]:
# --- Configuration ---
LONG_DIR = "field_temporal_datasets_long"
OUTPUT_DIR = "visualizations_global_trends"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_gr_homo():
    files = glob.glob(os.path.join(LONG_DIR, "*_long_features.csv"))
    if not files:
        files = glob.glob("*_long_features.csv")
        
    if not files:
        print(f"[ERROR] No files found.")
        return

    print(f"Aggregating data from {len(files)} fields...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Clean specific feature columns requested
        df = df.dropna(subset=['green_red_ratio', 'homogeneity', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid data extracted.")
        return

    # 1. Build Master Global Dataset
    master_df = pd.concat(all_data, ignore_index=True)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3])]

    # 2. Binary Classification (Healthy vs Infected)
    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    # 3. Aggregate Data (Average per Field, per Status, per Time Step)
    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'green_red_ratio': 'mean',
        'homogeneity': 'mean'
    }).reset_index()

    # --- Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    metrics = [
        {'col': 'green_red_ratio', 'ax_idx': 0, 'title': 'Global Biochemical Trajectory (Green/Red Ratio)', 'ylabel': 'Mean Green/Red Ratio per Field'},
        {'col': 'homogeneity', 'ax_idx': 1, 'title': 'Global Structural Trajectory (GLCM Homogeneity)', 'ylabel': 'Mean Homogeneity per Field'}
    ]

    for m in metrics:
        ax = axes[m['ax_idx']]
        
        for status, color in palette.items():
            subset = agg_df[(agg_df['Status'] == status) & (agg_df[m['col']].notna())]
            
            if subset.empty:
                continue

            # A. Plot the Spaghetti (Individual Field Trajectories)
            sns.lineplot(
                data=subset, 
                x='Days_Before_Sample', 
                y=m['col'],
                units='Field',
                estimator=None,
                color=color, 
                alpha=0.15,
                linewidth=1.5, 
                ax=ax
            )

            # B. Plot the LOESS Trendline (Global Smoothed Trend)
            sns.regplot(
                data=subset, 
                x='Days_Before_Sample', 
                y=m['col'],
                scatter=False,
                lowess=True,        # Ensure statsmodels is installed to use this
                color=color,
                line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
                ax=ax
            )

        # Formatting
        ax.set_title(m['title'], fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=13)
        ax.set_ylabel(m['ylabel'], fontsize=13)
        ax.grid(True, linestyle=':', alpha=0.7)

        # Reverse X-axis so time flows chronologically (e.g., 50 days to 0 days)
        current_xlim = ax.get_xlim()
        if current_xlim[0] < current_xlim[1]:
            ax.set_xlim(current_xlim[1], current_xlim[0])

    # 4. Custom Legend
    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    axes[0].legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend', 'Individual Field Trajectory'], 
        loc='best', fontsize=11, framealpha=0.9
    )
    
    if axes[1].get_legend(): axes[1].get_legend().remove()

    plt.suptitle("Global Spatio-Temporal Analysis (LOESS Smoothing)", fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_gr_homo.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_gr_homo()

Aggregating data from 24 fields...
Generated Global Spaghetti Plot: visualizations_global_trends\global_spaghetti_loess_gr_homo.png


# 🛰️ Planet API Quick-Search & Ordering for 2025 Fields
This block configures the Planet API credentials and sets up a pipeline to search for high-resolution PlanetScope scenes in 2025. It targets only fields sampled in 2025, calculates spatial geometries, queries for low-cloud imagery, and prepares a synchronized Planet order payload.


In [5]:
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"

HEADERS = {"Content-Type": "application/json"}
AUTH = (PLANET_API_KEY, "")

EXTRACTED_LAYERS_DIR = "extracted_layers"

# Assuming sample_dates is defined elsewhere in your environment
# sample_dates = {"Field_A": "15/05/2025", "Field_B": "20/06/2025"}

def place_global_2025_order():
    # Modification 1: Isolate fields from 2025
    target_fields = {k: v for k, v in sample_dates.items() if pd.to_datetime(v, dayfirst=True).year == 2025}
    
    print(f"Scanning {len(target_fields)} fields to build 1 global order...\n")

    global_item_ids = set()
    global_geometries = [] 
    processed_fields = 0

    for field_name, s_date_str in target_fields.items():
        sample_date = pd.to_datetime(s_date_str, dayfirst=True)
        start_date = sample_date - timedelta(days=56)
        
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")
        if not os.path.exists(gpkg_path):
            continue

        gdf_layer = gpd.read_file(gpkg_path)
        if gdf_layer.crs is None or gdf_layer.crs.to_epsg() != 4326:
            gdf_layer = gdf_layer.to_crs(epsg=4326)
            
        bbox = gdf_layer.total_bounds 
        field_box = box(*bbox)
        bbox_geom = mapping(field_box)

        # Modification 2: Search PSScene instead of REOrthoTile
        search_request = {
            "item_types": ["PSScene"], 
            "filter": {
                "type": "AndFilter",
                "config": [
                    {"type": "GeometryFilter", "field_name": "geometry", "config": bbox_geom},
                    {"type": "DateRangeFilter", "field_name": "acquired", "config": {
                        "gte": start_date.strftime("%Y-%m-%dT00:00:00.000Z"),
                        "lte": sample_date.strftime("%Y-%m-%dT23:59:59.999Z")
                    }},
                    {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lt": 0.50}} 
                ]
            }
        }

        res = requests.post(SEARCH_URL, json=search_request, headers=HEADERS, auth=AUTH)
        if res.status_code != 200:
            continue
            
        features = res.json().get('features', [])
        features.sort(key=lambda x: pd.to_datetime(x['properties']['acquired']))
        
        # Temporal Pruning
        selected_item_ids = []
        selected_dates = []
        
        for feat in features:
            acq_dt = pd.to_datetime(feat['properties']['acquired']).tz_localize(None)
            
            if not selected_dates:
                selected_dates.append(acq_dt)
                selected_item_ids.append(feat['id'])
            else:
                if (acq_dt - selected_dates[-1]).days >= 4:
                    selected_dates.append(acq_dt)
                    selected_item_ids.append(feat['id'])

        if len(selected_item_ids) >= 2:
            global_item_ids.update(selected_item_ids)
            global_geometries.append(field_box)
            processed_fields += 1
            print(f"  [ADDED] {field_name}: {len(selected_item_ids)} mathematically valid scenes.")

    if not global_item_ids:
        print("\nNo fields met the strict PSScene criteria for 2025. No order placed.")
        return

    print("\nValidating and dissolving geometries to prevent self-intersections...")
    merged_geometry = unary_union(global_geometries)
    valid_aoi_geojson = mapping(merged_geometry)

    print(f"\nConsolidated Order Summary:")
    print(f" - Fields included: {processed_fields}")
    print(f" - Total unique scenes: {len(global_item_ids)}")
    
    # Modification 3: Order 8-band Surface Reflectance with Usable Data Mask
    order_payload = {
        "name": "Global_2025_Thesis_Order",
        "products": [
            {
                "item_ids": list(global_item_ids),
                "item_type": "PSScene",
                "product_bundle": "analytic_8b_sr_udm2"
            }
        ],
        "tools": [
            {
                "clip": {
                    "aoi": valid_aoi_geojson 
                }
            }
        ]
    }

    print("\nSubmitting 1 global order to Planet API...")
    order_res = requests.post(ORDERS_URL, json=order_payload, headers=HEADERS, auth=AUTH)
    
    if order_res.status_code == 202:
        order_id = order_res.json().get('id')
        print(f"[SUCCESS] Order Placed! Order ID: {order_id}")
    else:
        print(f"[ORDER ERROR] Failed to place order: {order_res.status_code} - {order_res.text}")
        
if __name__ == "__main__":
    place_global_2025_order()

Scanning 11 fields to build 1 global order...

  [ADDED] Broomrape_Sampling_Gonen_2025-06-30: 12 mathematically valid scenes.
  [ADDED] Monitor_GDD562_Broomrape_2025-05-25: 11 mathematically valid scenes.
  [ADDED] Monitor_GDD714_Broomrape_2025-06-05: 11 mathematically valid scenes.
  [ADDED] Havat_mataim_7.7.2025: 12 mathematically valid scenes.
  [ADDED] manara_160725: 12 mathematically valid scenes.
  [ADDED] rimonim_100725: 13 mathematically valid scenes.
  [ADDED] Sasa_7_3.8.25: 12 mathematically valid scenes.
  [ADDED] Kfar_sald_shaked_east_240725: 13 mathematically valid scenes.
  [ADDED] Shamir_har_100725: 13 mathematically valid scenes.
  [ADDED] hazorea_100825: 13 mathematically valid scenes.
  [ADDED] gonen_maagar_220725: 12 mathematically valid scenes.

Validating and dissolving geometries to prevent self-intersections...

Consolidated Order Summary:
 - Fields included: 11
 - Total unique scenes: 74

Submitting 1 global order to Planet API...
[SUCCESS] Order Placed! Order I

# 📥 Planet API Order Download (2025 Datasets)
This block monitors and downloads the Planet order for the 2025 high-resolution imagery. It polls the Orders API until the order is successfully prepared, then streams the raw `.tif` files into `data/raw_chips_2025` for processing.


In [6]:

# --- Configuration ---
load_dotenv()
PLANET_API_KEY = os.getenv("PLANET_API_KEY")

# Updated Order ID from the 2025 PSScene execution
ORDER_ID = "28274c24-9597-4318-939c-b507c366ae1b" 
DOWNLOAD_DIR = "data/raw_chips_2025"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

ORDERS_URL = f"https://api.planet.com/compute/ops/orders/v2/{ORDER_ID}"
AUTH = (PLANET_API_KEY, "")

def download_planet_order():
    print(f"Checking status for Order: {ORDER_ID}")
    
    while True:
        res = requests.get(ORDERS_URL, auth=AUTH)
        if res.status_code != 200:
            print(f"[ERROR] Could not fetch order status: {res.status_code} - {res.text}")
            return
            
        order_data = res.json()
        state = order_data.get('state')
        
        print(f"Current Status: {state.upper()}")
        
        if state == 'success':
            break
        elif state in ['failed', 'partial']:
            print("[WARNING] Order did not complete fully. Check Planet UI for details.")
            if state == 'failed':
                return
            break 
        else:
            print("  -> Order is still processing. Waiting 30 seconds...")
            time.sleep(30)

    results = order_data.get('_links', {}).get('results', [])
    if not results:
        print("No download links found in the order response.")
        return

    # Because analytic_8b_sr_udm2 was ordered, this will download both the 
    # 8-band spectral data (*_SR.tif) and the usable data mask (*_udm2.tif).
    tif_files = [r for r in results if r['name'].endswith('.tif')]
    
    print(f"\nFound {len(tif_files)} .tif files ready for download.")
    
    downloaded_count = 0
    for file_info in tif_files:
        download_url = file_info['location']
        
        base_filename = os.path.basename(file_info['name'])
        save_path = os.path.join(DOWNLOAD_DIR, base_filename)
        
        if os.path.exists(save_path):
            print(f"  [SKIP] Already exists: {base_filename}")
            downloaded_count += 1
            continue
            
        print(f"  Downloading: {base_filename} ...", end="", flush=True)
        
        try:
            dl_res = requests.get(download_url, stream=True)
            dl_res.raise_for_status()
            
            with open(save_path, 'wb') as f:
                for chunk in dl_res.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(" Done.")
            downloaded_count += 1
            
        except Exception as e:
            print(f" FAILED. Error: {e}")

    print(f"\n[COMPLETE] Successfully secured {downloaded_count}/{len(tif_files)} files in {DOWNLOAD_DIR}/")

if __name__ == "__main__":
    download_planet_order()

Checking status for Order: 28274c24-9597-4318-939c-b507c366ae1b
Current Status: SUCCESS

Found 148 .tif files ready for download.
  Downloading: 20250609_084834_62_254a_3B_udm2_clip.tif ... Done.
  Downloading: 20250609_084834_62_254a_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Downloading: 20250617_083807_38_2409_3B_udm2_clip.tif ... Done.
  Downloading: 20250617_083807_38_2409_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Downloading: 20250714_084904_03_254a_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Downloading: 20250714_084904_03_254a_3B_udm2_clip.tif ... Done.
  Downloading: 20250523_084804_36_24e9_3B_udm2_clip.tif ... Done.
  Downloading: 20250523_084804_36_24e9_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Downloading: 20250709_084942_69_254a_3B_udm2_clip.tif ... Done.
  Downloading: 20250709_084942_69_254a_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Downloading: 20250526_083655_82_2515_3B_udm2_clip.tif ... Done.
  Downloading: 20250526_083655_82_2515_3B_AnalyticMS_SR_8b_clip.tif ... Done.
  Down

# 🧬 Pixel-Level Feature Extraction & Long-Format Construction (2025 Fields)
This block extracts pixel-level multi-spectral indices (NDVI, GNDVI, NDRE, Green/Red Ratio) and textural GLCM features (Entropy, Homogeneity, ASM) from the downloaded 2025 satellite imagery, then compiles them into long-format temporal datasets linked to the 2025 ground truth points.


In [3]:
# --- Constants & Paths ---
RAW_CHIPS_DIR = "data/raw_chips_2025" 
CLEAN_TABLES_DIR = "clean_tables"
EXTRACTED_LAYERS_DIR = "extracted_layers"
OUTPUT_DIR = "field_temporal_datasets_long_2025" 

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_valid_chips_for_field(field_gdf, chips_dir):
    valid_chips = []
    minx, miny, maxx, maxy = field_gdf.total_bounds
    
    centroid = field_gdf.geometry.unary_union.centroid
    cx, cy = centroid.x, centroid.y
    
    if not os.path.exists(chips_dir):
        return valid_chips

    for f in os.listdir(chips_dir):
        if not f.endswith('.tif') or 'udm' in f.lower() or 'sr' not in f.lower():
            continue

        chip_path = os.path.join(chips_dir, f)
        
        try:
            with rasterio.open(chip_path) as src:
                transformer = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                t_minx, t_miny = transformer.transform(minx, miny)
                t_maxx, t_maxy = transformer.transform(maxx, maxy)
                
                intersects = not (t_minx > src.bounds.right or t_maxx < src.bounds.left or
                                  t_miny > src.bounds.top or t_maxy < src.bounds.bottom)
                
                if intersects:
                    cx_proj, cy_proj = transformer.transform(cx, cy)
                    try:
                        py, px = src.index(cx_proj, cy_proj)
                        
                        val = src.read(8, window=Window(px, py, 1, 1)) 
                        
                        if val.size > 0 and val[0, 0] > 0:
                            # Using your upgraded flexible regex
                            match = re.search(r'(20\d{2}-?\d{2}-?\d{2})', f)
                            if match:
                                date_str = match.group(1)
                                acq_date = pd.to_datetime(date_str)
                                valid_chips.append({'path': chip_path, 'date': acq_date})
                    except Exception:
                        pass 
        except Exception:
            pass 
            
    return valid_chips

def extract_features_at_point(chip_path, lon, lat, use_1pixel=False):
    """
    Extracts high-resolution satellite features at a specific coordinate.
    If use_1pixel is True, spectral features are extracted strictly from the 1x1 pixel.
    GLCM texture calculations are always computed over a 3x3 surrounding window.
    """
    try:
        with rasterio.open(chip_path) as src:
            transformer = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            x_proj, y_proj = transformer.transform(lon, lat)
            
            if not ((src.bounds.left - 100) <= x_proj <= (src.bounds.right + 100) and 
                    (src.bounds.bottom - 100) <= y_proj <= (src.bounds.top + 100)):
                return None, "WRONG_FIELD"
                
            py, px = src.index(x_proj, y_proj)
            
            # Ensure safe boundary buffers for both windows
            if py < 1 or px < 1 or py >= src.height - 1 or px >= src.width - 1:
                return None, "EDGE_PROXIMITY"
            
            # --- 1. SPECTRAL FEATURE EXTRACTION ---
            if use_1pixel:
                # Extract strictly from the single 1x1 pixel containing the coordinate
                spectral_window = Window(px, py, 1, 1)
                spectral_data = src.read(window=spectral_window).astype(float)
                
                if spectral_data.shape[0] < 8:
                    return None, "SENSOR_MISMATCH"
                
                # Retrieve scalar values directly (no averaging required)
                green = spectral_data[3][0, 0]
                red = spectral_data[5][0, 0]
                red_edge = spectral_data[6][0, 0]
                nir = spectral_data[7][0, 0]
                
                if nir == 0:
                    return None, "NODATA_COLLAR"
                
                mean_green = green
                
                denom_ndvi = nir + red
                mean_ndvi = (nir - red) / denom_ndvi if denom_ndvi != 0 else np.nan

                denom_gndvi = nir + green
                mean_gndvi = (nir - green) / denom_gndvi if denom_gndvi != 0 else np.nan

                mean_gr_ratio = green / red if red != 0 else np.nan

                denom_ndre = nir + red_edge
                mean_ndre = (nir - red_edge) / denom_ndre if denom_ndre != 0 else np.nan
            else:
                # Standard 3x3 pixel window and spatial averages (original method)
                window = Window(px - 1, py - 1, 3, 3)
                data = src.read(window=window).astype(float)
                
                if data.shape[0] < 8:
                    return None, "SENSOR_MISMATCH"
                    
                green, red, red_edge, nir = data[3], data[5], data[6], data[7]
                
                if np.all(nir == 0):
                    return None, "NODATA_COLLAR"

                mean_green = np.nanmean(green)
                
                denom_ndvi = nir + red
                ndvi_arr = np.divide((nir - red), denom_ndvi, out=np.full_like(nir, np.nan), where=denom_ndvi!=0)
                mean_ndvi = np.nanmean(ndvi_arr)
                std_ndvi = np.nanstd(ndvi_arr)  # Calculates standard deviation over the 3x3 window

                denom_gndvi = nir + green
                gndvi_arr = np.divide((nir - green), denom_gndvi, out=np.full_like(nir, np.nan), where=denom_gndvi!=0)
                mean_gndvi = np.nanmean(gndvi_arr)

                gr_ratio_arr = np.divide(green, red, out=np.full_like(green, np.nan), where=red!=0)
                mean_gr_ratio = np.nanmean(gr_ratio_arr)

                denom_ndre = nir + red_edge
                ndre_arr = np.divide((nir - red_edge), denom_ndre, out=np.full_like(nir, np.nan), where=denom_ndre!=0)
                mean_ndre = np.nanmean(ndre_arr)

            # --- 2. TEXTURE FEATURE EXTRACTION (Always 3x3 window) ---
            texture_window = Window(px - 1, py - 1, 3, 3)
            texture_data = src.read(window=texture_window).astype(float)
            nir_texture = texture_data[7]
            
            nir_8bit = np.clip(nir_texture / 10000.0 * 255, 0, 255).astype(np.uint8)
            angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
            glcm = graycomatrix(nir_8bit, distances=[1], angles=angles, symmetric=True, normed=True)
            
            homo = np.mean(graycoprops(glcm, 'homogeneity'))
            asm = np.mean(graycoprops(glcm, 'ASM'))
            
            glcm_avg = np.mean(glcm, axis=3)[:, :, 0]
            entropy = -np.sum(glcm_avg[glcm_avg > 0] * np.log2(glcm_avg[glcm_avg > 0]))

            features = {
                'green': mean_green, 
                'ndvi': mean_ndvi, 
                'ndvi_std': std_ndvi,  # Added new column here
                'gndvi': mean_gndvi, 
                'green_red_ratio': mean_gr_ratio, 
                'ndre': mean_ndre,
                'homogeneity': homo, 
                'entropy': entropy, 
                'asm': asm
            }
            return features, "SUCCESS"
            
    except Exception as e:
        return None, f"EXCEPTION: {str(e)}"
        
def build_long_format_datasets():
    target_fields = {k: v for k, v in sample_dates.items() if pd.to_datetime(v, dayfirst=True).year == 2025}
    print(f"Initiating Long Format extraction for {len(target_fields)} fields...\n")

    for field_name, s_date_str in target_fields.items():
        field_records = [] 
        sample_date = pd.to_datetime(s_date_str, dayfirst=True)
        
        # Define strict temporal boundary
        start_date = sample_date - timedelta(days=60)
        
        csv_path = os.path.join(CLEAN_TABLES_DIR, f"{field_name}.csv")
        gpkg_path = os.path.join(EXTRACTED_LAYERS_DIR, f"{field_name}.gpkg")

        if not all(os.path.exists(p) for p in [csv_path, gpkg_path]):
            print(f"[SKIP] Missing core files for {field_name}")
            continue

        df_table = pd.read_csv(csv_path)
        gdf_layer = gpd.read_file(gpkg_path).to_crs(epsg=4326)
        id_col = 'id' if 'id' in df_table.columns else df_table.columns[0]
        
        if id_col not in gdf_layer.columns:
            if len(gdf_layer) == len(df_table):
                gdf_layer[id_col] = df_table[id_col].values
            else:
                continue
                
        print(f"--- Processing Field: {field_name} ---")
        
        # Enforce 1-pixel extraction ONLY for GDD714 and GDD562 fields
        use_1pixel = field_name in [
            "Monitor_GDD714_Broomrape_2025-06-05", 
            "Monitor_GDD562_Broomrape_2025-05-25"
        ]
        
        valid_chips = get_valid_chips_for_field(gdf_layer, RAW_CHIPS_DIR)
        
        if not valid_chips:
            print(f"  -> [WARNING] No valid overlapping images found in directory. Skipping field.")
            continue
            
        unique_dates = sorted(list(set([c['date'] for c in valid_chips])))
        
        # --- NEW: STRICT TEMPORAL ENFORCEMENT ---
        # Purge any dates > 60 days before sample, or occurring after sample
        valid_temporal_dates = [d for d in unique_dates if start_date <= d <= sample_date]
        
        if not valid_temporal_dates:
            print(f"  -> [WARNING] No images fall within the 60-day pre-sample window. Skipping field.")
            continue
        
        selected_dates = []
        selected_dates.append(valid_temporal_dates[0])
        for d in valid_temporal_dates[1:]:
            if (d - selected_dates[-1]).days >= 4:
                selected_dates.append(d)
        
        if len(selected_dates) < 2:
            print(f"  -> [WARNING] Not enough temporally spaced images (< 2) in 60-day window. Skipping field.")
            continue
            
        date_to_chips = {d: [] for d in selected_dates}
        for c in valid_chips:
            if c['date'] in selected_dates:
                date_to_chips[c['date']].append(c['path'])
        
        for idx, row in df_table.iterrows():
            pt_id = row[id_col]
            aleket = row['Aleket_cnt']
            
            pt_geom_series = gdf_layer[gdf_layer[id_col] == pt_id].geometry
            if pt_geom_series.empty:
                continue
            lat, lon = pt_geom_series.iloc[0].y, pt_geom_series.iloc[0].x
            
            for i, d in enumerate(selected_dates, start=1):
                date_str = d.strftime('%d/%m/%Y')
                record = {
                    'image': i, 'Point_ID': pt_id, 'Aleket_cnt': aleket,
                    'lat': lat, 'long': lon, 'date': date_str,
                    'Days_Before_Sample': (sample_date - d).days,
                    'ndvi': np.nan, 
                    'ndvi_std': np.nan,  # Added to column order directly after 'ndvi'
                    'gndvi': np.nan, 
                    'green_red_ratio': np.nan,
                    'ndre': np.nan, 
                    'entropy': np.nan, 
                    'green': np.nan,
                    'homogeneity': np.nan, 
                    'asm': np.nan
                }
                
                chip_paths = date_to_chips[d]
                success = False
                edge_proximity = False
                nodata_count = 0
                
                for chip_path in chip_paths:
                    features, error_msg = extract_features_at_point(
                        chip_path, lon, lat, use_1pixel=use_1pixel
                    )    
                    if features:
                        record.update(features)
                        success = True
                        break 
                    else:
                        if error_msg == "EDGE_PROXIMITY": edge_proximity = True
                        elif error_msg == "NODATA_COLLAR": nodata_count += 1
                            
                if not success:
                    if edge_proximity:
                        pass # Suppressed per previous logic
                    elif nodata_count > 0:
                        pass # Suppressed per previous logic
            
                field_records.append(record)

        if field_records:
            final_df = pd.DataFrame(field_records)
            final_df = final_df.sort_values(by=['image', 'Point_ID'])
            out_path = os.path.join(OUTPUT_DIR, f"{field_name}_long_features.csv")
            final_df.to_csv(out_path, index=False)
            print(f"  -> Saved {len(final_df)} observation rows.")
        else:
            print(f"  -> No data generated.")

if __name__ == "__main__":
    build_long_format_datasets()

Initiating Long Format extraction for 11 fields...

--- Processing Field: Broomrape_Sampling_Gonen_2025-06-30 ---
  -> Saved 2860 observation rows.
--- Processing Field: Monitor_GDD562_Broomrape_2025-05-25 ---
  -> Saved 216 observation rows.
--- Processing Field: Monitor_GDD714_Broomrape_2025-06-05 ---
  -> Saved 1963 observation rows.
--- Processing Field: Havat_mataim_7.7.2025 ---
  -> Saved 2004 observation rows.
--- Processing Field: manara_160725 ---
  -> Saved 1781 observation rows.
--- Processing Field: rimonim_100725 ---
  -> Saved 1885 observation rows.
--- Processing Field: Sasa_7_3.8.25 ---
  -> Saved 2520 observation rows.
--- Processing Field: Kfar_sald_shaked_east_240725 ---
  -> Saved 2743 observation rows.
--- Processing Field: Shamir_har_100725 ---
  -> Saved 1573 observation rows.
--- Processing Field: hazorea_100825 ---
  -> Saved 2639 observation rows.
--- Processing Field: gonen_maagar_220725 ---
  -> Saved 1846 observation rows.


# 📊 Global 2025 Trajectory Trends: NDVI Spaghetti Plot
This block aggregates all 2025 longitudinal datasets and generates global spaghetti plots for NDVI. It displays individual field trajectory lines alongside a LOESS-smoothed trend, visualizing the overall vegetation vigor over time.


In [4]:

# --- Configuration ---
LONG_DIR_2025 = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_global_trends_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_ndvi_2025():
    # Find all 2025 longitudinal datasets
    files = glob.glob(os.path.join(LONG_DIR_2025, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR_2025}.")
        return

    print(f"Aggregating data from {len(files)} fields for 2025 NDVI analysis...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Clean specific feature columns required for this plot
        df = df.dropna(subset=['ndvi', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid NDVI data extracted from the CSVs.")
        return

    # 1. Build Master Global Dataset
    master_df = pd.concat(all_data, ignore_index=True)
    
    # Optional: ensure Aleket counts are within standard biological ranges 
    # (can adjust if your 2025 data includes higher counts)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3, 4, 5])]

    # 2. Binary Classification (Healthy vs Infected)
    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    # 3. Aggregate Data (Average per Field, per Status, per Time Step)
    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'ndvi': 'mean'
    }).reset_index()

    # --- Visualization ---
    fig, ax = plt.subplots(figsize=(12, 8))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    for status, color in palette.items():
        subset = agg_df[(agg_df['Status'] == status) & (agg_df['ndvi'].notna())]
        
        if subset.empty:
            continue

        # A. Plot the Spaghetti (Individual Field Trajectories)
        sns.lineplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='ndvi',
            units='Field',
            estimator=None,
            color=color, 
            alpha=0.20,         # Slightly more opaque to see 2025 fields clearly
            linewidth=1.5, 
            ax=ax
        )

        # B. Plot the LOESS Trendline (Global Smoothed Trend)
        sns.regplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='ndvi',
            scatter=False,
            lowess=True,        
            color=color,
            line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
            ax=ax
        )

    # Formatting
    ax.set_title("2025 Global Trajectory: Canopy Vigor (NDVI)", fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    ax.set_ylabel("Mean NDVI per Field", fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.7)

    # Reverse X-axis so time flows chronologically (e.g., 60 days down to 0 days)
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])

    # 4. Custom Legend
    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    ax.legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend'], 
        loc='best', fontsize=12, framealpha=0.9
    )

    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_ndvi_2025.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_ndvi_2025()

Aggregating data from 11 fields for 2025 NDVI analysis...
Generated Global Spaghetti Plot: visualizations_global_trends_2025\global_spaghetti_loess_ndvi_2025.png


# 📊 Global 2025 Trajectory Trends: NDRE Spaghetti Plot
This block aggregates all 2025 datasets to visualize global spaghetti plots for NDRE (Normalized Difference Red Edge), tracking early chlorophyll and structural crop stress trends.


In [5]:

# --- Configuration ---
LONG_DIR_2025 = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_global_trends_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_ndre_2025():
    files = glob.glob(os.path.join(LONG_DIR_2025, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR_2025}.")
        return

    print(f"Aggregating data from {len(files)} fields for 2025 NDRE analysis...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        df = df.dropna(subset=['ndre', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid NDRE data extracted from the CSVs.")
        return

    master_df = pd.concat(all_data, ignore_index=True)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3, 4, 5])]

    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'ndre': 'mean'
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 8))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    for status, color in palette.items():
        subset = agg_df[(agg_df['Status'] == status) & (agg_df['ndre'].notna())]
        
        if subset.empty:
            continue

        sns.lineplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='ndre',
            units='Field',
            estimator=None,
            color=color, 
            alpha=0.20,         
            linewidth=1.5, 
            ax=ax
        )

        sns.regplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='ndre',
            scatter=False,
            lowess=True,        
            color=color,
            line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
            ax=ax
        )

    ax.set_title("2025 Global Trajectory: NDRE", fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    ax.set_ylabel("Mean NDRE per Field", fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.7)

    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])

    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    ax.legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend'], 
        loc='best', fontsize=12, framealpha=0.9
    )

    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_ndre_2025.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_ndre_2025()

Aggregating data from 11 fields for 2025 NDRE analysis...
Generated Global Spaghetti Plot: visualizations_global_trends_2025\global_spaghetti_loess_ndre_2025.png


# 📊 Global 2025 Trajectory Trends: GLCM Homogeneity Spaghetti Plot
This block aggregates all 2025 longitudinal datasets to generate a global spaghetti plot for GLCM Homogeneity, showing how spatial texture uniformity evolves across healthy and infested crop populations.


In [6]:


# --- Configuration ---
LONG_DIR_2025 = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_global_trends_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_homo_2025():
    files = glob.glob(os.path.join(LONG_DIR_2025, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR_2025}.")
        return

    print(f"Aggregating data from {len(files)} fields for 2025 GLCM Homogeneity analysis...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        df = df.dropna(subset=['homogeneity', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid GLCM Homogeneity data extracted from the CSVs.")
        return

    master_df = pd.concat(all_data, ignore_index=True)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3, 4, 5])]

    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'homogeneity': 'mean'
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 8))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    for status, color in palette.items():
        subset = agg_df[(agg_df['Status'] == status) & (agg_df['homogeneity'].notna())]
        
        if subset.empty:
            continue

        sns.lineplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='homogeneity',
            units='Field',
            estimator=None,
            color=color, 
            alpha=0.20,         
            linewidth=1.5, 
            ax=ax
        )

        sns.regplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='homogeneity',
            scatter=False,
            lowess=True,        
            color=color,
            line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
            ax=ax
        )

    ax.set_title("2025 Global Trajectory: GLCM Homogeneity", fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    ax.set_ylabel("Mean GLCM Homogeneity per Field", fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.7)

    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])

    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    ax.legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend'], 
        loc='best', fontsize=12, framealpha=0.9
    )

    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_homo_2025.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_homo_2025()

Aggregating data from 11 fields for 2025 GLCM Homogeneity analysis...
Generated Global Spaghetti Plot: visualizations_global_trends_2025\global_spaghetti_loess_homo_2025.png


# 📊 Global 2025 Trajectory Trends: GNDVI Spaghetti Plot
This block aggregates all 2025 datasets to generate spaghetti plots for GNDVI (Green NDVI), tracking chlorophyll content and photosynthetic activity leading up to sampling.


In [7]:


# --- Configuration ---
LONG_DIR_2025 = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_global_trends_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_global_spaghetti_gndvi_2025():
    files = glob.glob(os.path.join(LONG_DIR_2025, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No files found in {LONG_DIR_2025}.")
        return

    print(f"Aggregating data from {len(files)} fields for 2025 GNDVI analysis...")
    all_data = []

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        df = df.dropna(subset=['gndvi', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            continue
            
        df['Field'] = field_name
        all_data.append(df)

    if not all_data:
        print("[ERROR] No valid GNDVI data extracted from the CSVs.")
        return

    master_df = pd.concat(all_data, ignore_index=True)
    master_df = master_df[master_df['Aleket_cnt'].isin([0, 1, 2, 3, 4, 5])]

    master_df['Status'] = np.where(
        master_df['Aleket_cnt'] == 0, 
        'Healthy (Aleket = 0)', 
        'Infected (Aleket > 0)'
    )

    agg_df = master_df.groupby(['Field', 'Status', 'Days_Before_Sample']).agg({
        'gndvi': 'mean'
    }).reset_index()

    fig, ax = plt.subplots(figsize=(12, 8))
    palette = {'Healthy (Aleket = 0)': '#2CA02C', 'Infected (Aleket > 0)': '#D62728'}

    for status, color in palette.items():
        subset = agg_df[(agg_df['Status'] == status) & (agg_df['gndvi'].notna())]
        
        if subset.empty:
            continue

        sns.lineplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='gndvi',
            units='Field',
            estimator=None,
            color=color, 
            alpha=0.20,         
            linewidth=1.5, 
            ax=ax
        )

        sns.regplot(
            data=subset, 
            x='Days_Before_Sample', 
            y='gndvi',
            scatter=False,
            lowess=True,        
            color=color,
            line_kws={'linewidth': 4, 'solid_capstyle': 'round'}, 
            ax=ax
        )

    ax.set_title("2025 Global Trajectory: GNDVI", fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=14)
    ax.set_ylabel("Mean GNDVI per Field", fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.7)

    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])

    custom_lines = [
        Line2D([0], [0], color='#2CA02C', lw=4),
        Line2D([0], [0], color='#D62728', lw=4),
        Line2D([0], [0], color='#7F7F7F', lw=1.5, alpha=0.5)
    ]
    ax.legend(
        custom_lines, 
        ['Global Healthy Trend', 'Global Infected Trend'], 
        loc='best', fontsize=12, framealpha=0.9
    )

    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, "global_spaghetti_loess_gndvi_2025.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"Generated Global Spaghetti Plot: {out_path}")

if __name__ == "__main__":
    generate_global_spaghetti_gndvi_2025()

Aggregating data from 11 fields for 2025 GNDVI analysis...
Generated Global Spaghetti Plot: visualizations_global_trends_2025\global_spaghetti_loess_gndvi_2025.png


# 🧪 Individual Field Scatter Plots: 2025 NDVI Distribution
This block generates individual scatter plots of NDVI against days before sampling for each 2025 field, coloring ground-truth points by their final Aleket severity classes.


In [8]:


# --- Configuration ---
INPUT_DIR = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "ndvi_scatter_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_individual_scatter_plots():
    files = glob.glob(os.path.join(INPUT_DIR, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No CSV files found in {INPUT_DIR}.")
        return

    print(f"Generating scatter plots for {len(files)} fields...")

    # Standardized color palette
    palette = {
        0: '#2CA02C',  # Green
        1: '#FFD700',  # Yellow
        2: '#FF7F0E',  # Orange
        3: '#D62728'   # Red
    }

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Remove rows missing necessary data
        df = df.dropna(subset=['ndvi', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            print(f"  [SKIP] {field_name}: No valid NDVI data.")
            continue
            
        # Group any severity >= 3 into the maximum category for the red color mapping
        df['Severity'] = df['Aleket_cnt'].apply(lambda x: x if x <= 2 else 3)
            
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot the scatter distribution
        sns.scatterplot(
            data=df,
            x='Days_Before_Sample',
            y='ndvi',
            hue='Severity',
            palette=palette,
            s=70,               # Marker size
            alpha=0.75,         # Slight transparency to show overlapping points
            edgecolor='white',  # Clean border around points
            linewidth=0.5,
            ax=ax
        )
            
        # --- Formatting ---
        ax.set_title(f"NDVI Scatter Distribution: {field_name}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=12)
        ax.set_ylabel("NDVI", fontsize=12)
        
        # Reverse X-axis to display chronological time flow
        current_xlim = ax.get_xlim()
        if current_xlim[0] < current_xlim[1]:
            ax.set_xlim(current_xlim[1], current_xlim[0])
            
        # Custom Legend
        custom_lines = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#2CA02C', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFD700', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF7F0E', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728', markersize=10)
        ]
        
        # Override Seaborn's default legend
        if ax.get_legend() is not None:
            ax.get_legend().remove()
            
        ax.legend(
            custom_lines, 
            ['Healthy (0)', 'Low Infection (1)', 'Moderate Infection (2)', 'Severe Infection (≥3)'], 
            loc='best', 
            fontsize=10, 
            title="Aleket Severity",
            framealpha=0.9
        )
        
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        
        out_path = os.path.join(OUTPUT_DIR, f"{field_name}_ndvi_scatter.png")
        plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
        plt.close()
        
        print(f"  -> Saved: {out_path}")

if __name__ == "__main__":
    generate_individual_scatter_plots()

Generating scatter plots for 11 fields...
  -> Saved: ndvi_scatter_2025\Broomrape_Sampling_Gonen_2025-06-30_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\gonen_maagar_220725_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\Havat_mataim_7.7.2025_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\hazorea_100825_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\Kfar_sald_shaked_east_240725_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\manara_160725_ndvi_scatter.png
  [SKIP] Monitor_GDD562_Broomrape_2025-05-25: No valid NDVI data.
  [SKIP] Monitor_GDD714_Broomrape_2025-06-05: No valid NDVI data.
  -> Saved: ndvi_scatter_2025\rimonim_100725_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\Sasa_7_3.8.25_ndvi_scatter.png
  -> Saved: ndvi_scatter_2025\Shamir_har_100725_ndvi_scatter.png


# 🧪 Individual Field Scatter Plots: 2025 Green/Red Ratio Distribution
This block generates individual scatter plots of the Green/Red Ratio against days before sampling for each 2025 field, allowing us to inspect spatial-temporal shifts in chlorosis per severity class.


In [9]:

# --- Configuration ---
INPUT_DIR = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_scatter_gr_ratio_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_individual_scatter_plots_gr():
    files = glob.glob(os.path.join(INPUT_DIR, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No CSV files found in {INPUT_DIR}.")
        return

    print(f"Generating scatter plots for {len(files)} fields...")

    # Standardized color palette
    palette = {
        0: '#2CA02C',  # Green
        1: '#FFD700',  # Yellow
        2: '#FF7F0E',  # Orange
        3: '#D62728'   # Red
    }

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Remove rows missing necessary data
        df = df.dropna(subset=['green_red_ratio', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            print(f"  [SKIP] {field_name}: No valid Green/Red Ratio data.")
            continue
            
        # Group any severity >= 3 into the maximum category for the red color mapping
        df['Severity'] = df['Aleket_cnt'].apply(lambda x: x if x <= 2 else 3)
            
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot the scatter distribution
        sns.scatterplot(
            data=df,
            x='Days_Before_Sample',
            y='green_red_ratio',
            hue='Severity',
            palette=palette,
            s=70,               
            alpha=0.75,         
            edgecolor='white',  
            linewidth=0.5,
            ax=ax
        )
            
        # --- Formatting ---
        ax.set_title(f"Green/Red Ratio Scatter Distribution: {field_name}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=12)
        ax.set_ylabel("Green/Red Ratio", fontsize=12)
        
        # Reverse X-axis to display chronological time flow
        current_xlim = ax.get_xlim()
        if current_xlim[0] < current_xlim[1]:
            ax.set_xlim(current_xlim[1], current_xlim[0])
            
        # Custom Legend
        custom_lines = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#2CA02C', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFD700', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF7F0E', markersize=10),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728', markersize=10)
        ]
        
        # Override Seaborn's default legend
        if ax.get_legend() is not None:
            ax.get_legend().remove()
            
        ax.legend(
            custom_lines, 
            ['Healthy (0)', 'Low Infection (1)', 'Moderate Infection (2)', 'Severe Infection (≥3)'], 
            loc='best', 
            fontsize=10, 
            title="Aleket Severity",
            framealpha=0.9
        )
        
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        
        out_path = os.path.join(OUTPUT_DIR, f"{field_name}_gr_ratio_scatter.png")
        plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
        plt.close()
        
        print(f"  -> Saved: {out_path}")

if __name__ == "__main__":
    generate_individual_scatter_plots_gr()

Generating scatter plots for 11 fields...
  -> Saved: visualizations_scatter_gr_ratio_2025\Broomrape_Sampling_Gonen_2025-06-30_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\gonen_maagar_220725_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\Havat_mataim_7.7.2025_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\hazorea_100825_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\Kfar_sald_shaked_east_240725_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\manara_160725_gr_ratio_scatter.png
  [SKIP] Monitor_GDD562_Broomrape_2025-05-25: No valid Green/Red Ratio data.
  [SKIP] Monitor_GDD714_Broomrape_2025-06-05: No valid Green/Red Ratio data.
  -> Saved: visualizations_scatter_gr_ratio_2025\rimonim_100725_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\Sasa_7_3.8.25_gr_ratio_scatter.png
  -> Saved: visualizations_scatter_gr_ratio_2025\Shamir_har_100725_gr_ratio_sca

# 🧪 Individual Field Scatter Plots: 2025 GLCM Homogeneity Distribution
This block generates individual scatter plots of GLCM Homogeneity against days before sampling for each 2025 field to visualize spatial structural variation over time.


In [10]:

# --- Configuration ---
INPUT_DIR = "field_temporal_datasets_long_2025"
OUTPUT_DIR = "visualizations_scatter_homo_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_individual_scatter_plots_homo():
    files = glob.glob(os.path.join(INPUT_DIR, "*_long_features.csv"))
    
    if not files:
        print(f"[ERROR] No CSV files found in {INPUT_DIR}.")
        return

    print(f"Generating scatter plots for {len(files)} fields...")

    # Standardized color palette based on Aleket severity
    palette = {
        0: '#2CA02C',  # Green
        1: '#FFD700',  # Yellow
        2: '#FF7F0E',  # Orange
        3: '#D62728'   # Red
    }

    for f in files:
        field_name = os.path.basename(f).replace("_long_features.csv", "")
        df = pd.read_csv(f)
        
        # Remove rows missing necessary data
        df = df.dropna(subset=['homogeneity', 'Aleket_cnt', 'Days_Before_Sample'])
        if df.empty:
            print(f"  [SKIP] {field_name}: No valid Homogeneity data.")
            continue
            
        # Group any severity >= 3 into the maximum category for the red color mapping
        df['Severity'] = df['Aleket_cnt'].apply(lambda x: x if x <= 2 else 3)
            
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot the scatter distribution
        sns.scatterplot(
            data=df,
            x='Days_Before_Sample',
            y='homogeneity',
            hue='Severity',
            palette=palette,
            s=70,               
            alpha=0.75,         
            edgecolor='white',  
            linewidth=0.5,
            ax=ax
        )
            
        # --- Formatting ---
        ax.set_title(f"GLCM Homogeneity Scatter Distribution: {field_name}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Days Before Sampling (Chronological →)", fontsize=12)
        ax.set_ylabel("GLCM Homogeneity", fontsize=12)
        
        # Reverse X-axis to display chronological time flow
        current_xlim = ax.get_xlim()
        if current_xlim[0] < current_xlim[1]:
            ax.set_xlim(current_xlim[1], current_xlim[0])
            
        # Custom Legend
        custom_lines = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#2CA02C', markersize=8),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FFD700', markersize=8),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF7F0E', markersize=8),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#D62728', markersize=8)
        ]
        
        # Override Seaborn's default legend
        if ax.get_legend() is not None:
            ax.get_legend().remove()
            
        ax.legend(
            custom_lines, 
            ['Healthy (0)', 'Low Infection (1)', 'Moderate Infection (2)', 'Severe Infection (≥3)'], 
            loc='upper left', 
            bbox_to_anchor=(1.02, 1),
            fontsize=8, 
            title="Aleket Severity",
            title_fontsize=9,
            framealpha=0.9
        )
        
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        
        out_path = os.path.join(OUTPUT_DIR, f"{field_name}_homo_scatter.png")
        plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
        plt.close()
        
        print(f"  -> Saved: {out_path}")

if __name__ == "__main__":
    generate_individual_scatter_plots_homo()

Generating scatter plots for 11 fields...
  -> Saved: visualizations_scatter_homo_2025\Broomrape_Sampling_Gonen_2025-06-30_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\gonen_maagar_220725_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\Havat_mataim_7.7.2025_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\hazorea_100825_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\Kfar_sald_shaked_east_240725_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\manara_160725_homo_scatter.png
  [SKIP] Monitor_GDD562_Broomrape_2025-05-25: No valid Homogeneity data.
  [SKIP] Monitor_GDD714_Broomrape_2025-06-05: No valid Homogeneity data.
  -> Saved: visualizations_scatter_homo_2025\rimonim_100725_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\Sasa_7_3.8.25_homo_scatter.png
  -> Saved: visualizations_scatter_homo_2025\Shamir_har_100725_homo_scatter.png


In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
VIS_OUTPUT_DIR: str = "visualizations_temporal_std_trail"
os.makedirs(VIS_OUTPUT_DIR, exist_ok=True)

# Dataset Paths
csv_havat_gadash: str = r"field_temporal_datasets_long_2025\Monitor_GDD714_Broomrape_2025-06-05_long_features.csv"
csv_gonen: str = r"field_temporal_datasets_long_2025\Broomrape_Sampling_Gonen_2025-06-30_long_features.csv"

def generate_ndvi_std_trail_plot(csv_path: str, field_name: str) -> None:
    """
    Generates and saves a temporal NDVI trajectory plot where each line represents 
    a different Broomrape (Aleket) severity class, surrounded by a shaded band 
    representing the standard deviation (Std Trail).

    Parameters:
    -----------
    csv_path : str
        Path to the longitudinal long features CSV file.
    field_name : str
        Display name of the agricultural field.
    """
    # 1. Load and clean the temporal dataset
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['Days_Before_Sample', 'ndvi', 'Aleket_cnt'])
    
    # Cast to integer then string to ensure Seaborn treats it as a discrete category
    df['Aleket_cnt'] = df['Aleket_cnt'].astype(int).astype(str)
    
    # Strictly filter for standard Aleket scores '0' through '3'
    df = df[df['Aleket_cnt'].isin(['0', '1', '2', '3'])]
    
    if df.empty:
        print(f"  [SKIP] No valid data in 0-3 range for {field_name}")
        return

    # Define color scheme: '0': Green, '1': Yellow, '2': Orange, '3': Red
    aleket_palette = {
        '0': '#2CA02C',  # Green (Healthy)
        '1': '#FFD700',  # Yellow (Light)
        '2': '#FF7F0E',  # Orange (Medium)
        '3': '#D62728'   # Red (Severe)
    }

    # 2. Render plot with extra horizontal width to fit the outside legend
    plt.figure(figsize=(11, 6))
    
    # errorbar='sd' draws the standard deviation band around the mean line
    ax = sns.lineplot(
        data=df,
        x='Days_Before_Sample',
        y='ndvi',
        hue='Aleket_cnt',
        palette=aleket_palette,
        marker='o',
        markersize=8,
        linewidth=2.5,
        errorbar='sd'  # Renders the std trail around the mean line
    )
    
    # Reverse x-axis so the countdown flows chronologically (e.g., 56 days before -> 0)
    current_xlim = ax.get_xlim()
    if current_xlim[0] < current_xlim[1]:
        ax.set_xlim(current_xlim[1], current_xlim[0])
        
    # Styling and Labels
    plt.title(f"Temporal NDVI Trajectory by Infestation Severity (Standard Deviation Trail)\nField: {field_name}", 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Days Before Sampling (Chronological Countdown ->)", fontsize=12)
    plt.ylabel("NDVI (Physiological Vigor)", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Move the legend outside the plot area on the right-hand side and reduce its size
    handles, labels = ax.get_legend_handles_labels()
    legend_labels = [f"Score {l}" for l in labels]
    
    plt.legend(
        handles=handles, 
        labels=legend_labels, 
        title="Aleket Severity", 
        bbox_to_anchor=(1.04, 1), 
        loc='upper left', 
        frameon=True, 
        shadow=True, 
        fontsize=9, 
        title_fontsize=10
    )
    
    plt.tight_layout()
    
    # Save the output file
    file_suffix = field_name.lower().replace(" ", "_")
    out_path = os.path.join(VIS_OUTPUT_DIR, f"{file_suffix}_ndvi_std_trail.png")
    plt.savefig(out_path, dpi=300, facecolor='white', bbox_inches='tight')
    plt.close()
    
    print(f"  [OK] Saved standard deviation trail plot -> {out_path}")

# Run the plotting code for both fields
print("Generating NDVI standard deviation trail trajectories...")
generate_ndvi_std_trail_plot(csv_havat_gadash, "Havat Gadash")
generate_ndvi_std_trail_plot(csv_gonen, "Gonen")

Generating NDVI standard deviation trail trajectories...
  [OK] Saved standard deviation trail plot -> visualizations_temporal_std_trail\havat_gadash_ndvi_std_trail.png
  [OK] Saved standard deviation trail plot -> visualizations_temporal_std_trail\gonen_ndvi_std_trail.png
